# Adaptive Feature Redistribution on AGNews — Qwen3-14B-IT
## Steering Prompt V1: Topic-Only Boundary — Causal Screening + Held-Out Evaluation

The possible AGNews labels are **World, Sports, Business, and Technology** in the original condition; the target condition removes the selected `target_label` from the first instruction sentence and from every labeled example, leaving only the other three possible labels.

## Model and SAE

- Model: `Qwen/Qwen3-14B`
- Residual-stream layer: `20`
- SAE: `adamkarvonen/qwen3-14b-saes` / `saes_Qwen_Qwen3-14B_batch_top_k/resid_post_layer_20/trainer_0/ae.pt`
- SAE setting: Adam Karvonen Qwen3-14B residual-stream BatchTopK SAE, layer 20, trainer 0 (16k width, nominal L0 80)
- Provenance note: The SAE and instruction-tuned model use the same public Qwen3-14B checkpoint.

## Causal output-scoring prompt boundary

The causal output-scoring prompt ends exactly at the user-side `Topic:\n` boundary. The model's ordinary chat-template turn ending and assistant-generation prefix are deliberately removed.

This version combines the topic-only output boundary with independent causal-specificity screening and query-disjoint held-out evaluation; the activation-level AFR discovery procedure remains unchanged.

## Experiment design

- `target_label` is one of `{0, 1, 2, 3}`:
  - `0 = World`
  - `1 = Sports`
  - `2 = Business`
  - `3 = Technology`
- Every activation-analysis prompt contains exactly **8 labeled AGNews examples**. The final, scored item is always **Example 8**.
- **Original condition:** exactly two examples from each of the four labels.
- **Target condition:** the selected `target_label` is absent. For every final example belonging to a retained label, Example 8 is exactly one-to-one matched across conditions. Five of the seven context examples are also preserved; only the two context examples carrying the removed label are replaced. The resulting retained-label counts are balanced as `3/3/2`.
- `activation_type = "pre"` measures SAE encoder preactivations before the configured sparse gate; `activation_type = "post"` measures the corresponding post-gate activations.
- `frequency_filter = 0.01` retains only features that fired on at least 1% of the **original-condition final Example-8 instances whose label is `target_label`**. An instance counts as fired when the feature is active on at least one selected score token.
- `LAST_K_TOKENS` is an explicit list of offsets counted backward from the end of the extended Example-8 span. For this model, the span ends with `<|im_end|>`. Thus `-1` selects `<|im_end|>`, `-2` selects the immediately preceding token, and so forth. The default `[-2, -3, -4]` averages the three tokens immediately before the turn-ending token and excludes that token itself.
- The activation analysis identifies features that are selective for the removed label and tests whether their values increase on retained-label examples after that label disappears.
- The statistical analysis is a paired Wilcoxon signed-rank / rank-biserial volcano analysis.

## Causal AFR experiment

The causal analysis uses three query-disjoint stages. First, the notebook identifies 40 activation-level AFR features (top 10 per label) by requiring original preactivation specificity and the firing-frequency threshold, then ranking eligible features by their matched-pairs activation-level AFR. Second, every selected feature is tested one at a time on an independent **causal-screening** set containing only original four-label prompts. A feature is retained only when its original primary-label causal effect is positive and larger than its strongest original non-primary effect. Third, only screened-in features are evaluated on a separate **held-out** set of original and label-exclusion prompts.

The final causal bars are computed exclusively from held-out prompts:

- **green:** mean change in \(P(L)\) on held-out original prompts;
- **red:** strongest mean change in \(P(c)\), \(c\neq L\), on the same held-out original prompts;
- **blue:** strongest mean change in \(P(L')\), \(L'\neq L\), on held-out \(L\)-excluded prompts.

By default, activation discovery, causal screening, and held-out evaluation each use 100 final queries per label, and the three final-query pools are mutually disjoint. The causal output-scoring prompt ends exactly at the user-side `Topic:\n` boundary, and all probabilities are normalized over the same four AGNews label tokens.


In [ ]:
# ============================================================
# 0. Imports and all user-editable configuration
# ============================================================

import os
import re
import json
import math
import random
import hashlib
from typing import Any, Dict, List, Tuple, Optional, Sequence

import numpy as np
import pandas as pd
import torch

import plotly.graph_objects as go
from IPython.display import display

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from scipy.stats import wilcoxon, rankdata


# -----------------------------
# Reproducibility
# -----------------------------

SEED = 42


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# -----------------------------
# AGNews AFR configuration
# -----------------------------

LABEL_TO_WORD = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Technology",
}
WORD_TO_LABEL = {v: k for k, v in LABEL_TO_WORD.items()}
ALL_LABELS = list(LABEL_TO_WORD)

# Select the label that disappears in the target condition.
target_label = 0

# Choose which SAE quantity is analyzed throughout the notebook:
#   "pre"  = encoder preactivation before the configured sparse gate.
#   "post" = post-gate sparse activation.
activation_type = "pre"

# Volcano frequency filter. A feature is eligible for the rank-biserial plot only
# when it fired in at least this fraction of the ORIGINAL-condition final Example-8
# instances whose final label is target_label.
# Example: with 100 target-label instances, 0.01 means at least one instance.
frequency_filter = 0.05

if target_label not in LABEL_TO_WORD:
    raise ValueError(f"target_label must be one of {ALL_LABELS}; got {target_label}.")

activation_type = str(activation_type).strip().lower()
if activation_type not in {"pre", "post"}:
    raise ValueError(
        f"activation_type must be 'pre' or 'post'; got {activation_type!r}."
    )

frequency_filter = float(frequency_filter)
if not (0.0 <= frequency_filter <= 1.0):
    raise ValueError(
        f"frequency_filter must lie in [0, 1]; got {frequency_filter}."
    )

kept_labels = [label for label in ALL_LABELS if label != target_label]

ACTIVATION_VALUE_NAME = (
    "preactivation" if activation_type == "pre" else "post-threshold activation"
)
ACTIVATION_COLUMN_SUFFIX = (
    "preactivation" if activation_type == "pre" else "postactivation"
)
ACTIVATION_AXIS_LABEL = (
    "SAE preactivation" if activation_type == "pre" else "SAE post-threshold activation"
)

# Number of final Example-8 articles sampled from the AGNews test split for each label.
# Original prompts: 4 * NUM_FINAL_EXAMPLES_PER_LABEL.
# Target prompts:   3 * NUM_FINAL_EXAMPLES_PER_LABEL.
NUM_FINAL_EXAMPLES_PER_LABEL = 100

N_EXAMPLES_PER_PROMPT = 8
ORIGINAL_EXAMPLES_PER_LABEL = 2
MAX_ARTICLE_CHARS = 1000

# Explicit end-relative score-token offsets.
# INCLUDE_FINAL_TURN_END_IN_SCORE_SPAN=True first extends Example 8 through
# the model's following <|im_end|> token, so offsets are interpreted as:
#   -1 = <|im_end|>
#   -2 = token immediately before <|im_end|>
#   -3 = the preceding token, and so on.
# The default below therefore selects the three tokens before <|im_end|>
# and does NOT include the turn-ending token. Order does not affect the mean; selected
# token indices are internally sorted into left-to-right sequence order.
LAST_K_TOKENS = [-1, -2]
INCLUDE_FINAL_TURN_END_IN_SCORE_SPAN = True

if not isinstance(LAST_K_TOKENS, (list, tuple)) or len(LAST_K_TOKENS) == 0:
    raise ValueError(
        "LAST_K_TOKENS must be a non-empty list/tuple of negative integer "
        "offsets, e.g. [-2, -3, -4]."
    )

_normalized_last_k_tokens = []
for _offset in LAST_K_TOKENS:
    if isinstance(_offset, bool) or not isinstance(_offset, (int, np.integer)):
        raise TypeError(
            "Every LAST_K_TOKENS entry must be a negative integer; "
            f"got {_offset!r}."
        )
    _offset = int(_offset)
    if _offset >= 0:
        raise ValueError(
            "LAST_K_TOKENS uses offsets counted backward from the end, so every "
            f"entry must be negative; got {_offset}."
        )
    _normalized_last_k_tokens.append(_offset)

if len(set(_normalized_last_k_tokens)) != len(_normalized_last_k_tokens):
    raise ValueError(
        f"LAST_K_TOKENS contains duplicate offsets: {_normalized_last_k_tokens}."
    )

LAST_K_TOKENS = _normalized_last_k_tokens
del _normalized_last_k_tokens, _offset

TOPK_TARGET_SELECTIVE_FEATURES = 500
VOLCANO_P_THRESHOLD = 0.01
RUN_ANTIPODALITY_FOR_TOP_FEATURES = True

BATCH_SIZE = 1
PRINT_EVERY_BATCHES = 25

# -----------------------------
# Model and SAE configuration
# -----------------------------

MODEL_NAME = 'Qwen/Qwen3-14B'
MODEL_SHORT_NAME = 'Qwen3-14B-IT'
TARGET_LAYER = 20

SAE_BACKEND = 'qwen_batchtopk'
SAE_REPO_ID = 'adamkarvonen/qwen3-14b-saes'
SAE_FILENAME = 'saes_Qwen_Qwen3-14B_batch_top_k/resid_post_layer_20/trainer_0/ae.pt'
SAE_HYPERPARAMS_FILENAME = None
SAE_CONFIG_FILENAME = 'saes_Qwen_Qwen3-14B_batch_top_k/resid_post_layer_20/trainer_0/config.json'
SAE_WIDTH = 'width_16k'
SAE_AVERAGE_L0 = 'nominal_l0_80'
SAE_TOPK_K = 80
SAE_TRAINED_MODEL_NAME = 'Qwen/Qwen3-14B'
SAE_PROVENANCE = 'Adam Karvonen Qwen3-14B residual-stream BatchTopK SAE, layer 20, trainer 0 (16k width, nominal L0 80)'
SAE_TRANSFER_NOTE = 'The SAE and instruction-tuned model use the same public Qwen3-14B checkpoint.'

MODEL_TURN_END_TOKEN = '<|im_end|>'
TOKENIZER_NAME = 'Qwen/Qwen3-14B'
TOKENIZER_ADD_BOS_TOKEN = None
MODEL_SUPPORTS_ENABLE_THINKING = True
MODEL_TRUST_REMOTE_CODE = False

EXPECTED_MODEL_NUM_LAYERS = 40
EXPECTED_D_MODEL = 5120
EXPECTED_D_SAE = 16384

DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model/tokenizer from the local cache. Public SAE files may be
# downloaded anonymously when missing; the loader passes token=False directly.
# No authentication-discovery or token-management helper is used.
MODEL_LOCAL_FILES_ONLY = True
SAE_LOCAL_FILES_ONLY = False

print("Removed target label:", target_label, LABEL_TO_WORD[target_label])
print("Retained labels:", {label: LABEL_TO_WORD[label] for label in kept_labels})
print("activation_type:", activation_type, f"({ACTIVATION_VALUE_NAME})")
print("frequency_filter:", frequency_filter)
print("MODEL_NAME:", MODEL_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("SAE_BACKEND:", SAE_BACKEND)
print("SAE:", SAE_REPO_ID, SAE_FILENAME)
print("SAE provenance:", SAE_PROVENANCE)
print("SAE transfer note:", SAE_TRANSFER_NOTE)
print("DTYPE:", DTYPE)
print("LAST_K_TOKENS:", LAST_K_TOKENS)


## 1. Load Qwen3-14B-IT and the layer-20 SAE

The tokenizer and model are loaded directly from the local Hugging Face cache. The public SAE may be downloaded anonymously when absent from the cache; no authentication-discovery or token-management helper is included.

This version uses Adam Karvonen Qwen3-14B residual-stream BatchTopK SAE, layer 20, trainer 0 (16k width, nominal L0 80). The SAE and instruction-tuned model use the same public Qwen3-14B checkpoint. The notebook measures either encoder **preactivations** (`activation_type = "pre"`) or post-gate sparse activations (`activation_type = "post"`).


In [ ]:
# ============================================================
# 1. Direct cache-only tokenizer / model / SAE loading
# ============================================================


def load_tokenizer_direct():
    """Load the configured tokenizer directly, without authentication helpers."""
    print(f"Loading tokenizer: {TOKENIZER_NAME}")
    kwargs = {
        "pretrained_model_name_or_path": TOKENIZER_NAME,
        "use_fast": True,
        "trust_remote_code": MODEL_TRUST_REMOTE_CODE,
        "local_files_only": MODEL_LOCAL_FILES_ONLY,
    }
    if TOKENIZER_ADD_BOS_TOKEN is not None:
        kwargs["add_bos_token"] = bool(TOKENIZER_ADD_BOS_TOKEN)
    return AutoTokenizer.from_pretrained(**kwargs)


def load_model_direct(model_name: str):
    """Load the configured causal LM directly, without authentication helpers."""
    print(f"Loading model: {model_name}")
    model_obj = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=MODEL_TRUST_REMOTE_CODE,
        low_cpu_mem_usage=True,
        local_files_only=MODEL_LOCAL_FILES_ONLY,
    )
    model_obj.eval()
    for parameter in model_obj.parameters():
        parameter.requires_grad_(False)
    return model_obj


def _as_tensor(value) -> torch.Tensor:
    if isinstance(value, np.ndarray):
        value = torch.from_numpy(value)
    if torch.is_tensor(value):
        return value.detach()
    return torch.tensor(value)


def _expand_threshold(threshold: torch.Tensor, n_features: int) -> torch.Tensor:
    threshold = _as_tensor(threshold).reshape(-1)
    if threshold.numel() == 1:
        threshold = threshold.repeat(int(n_features))
    if threshold.numel() != int(n_features):
        raise ValueError(
            f"Threshold has {threshold.numel()} values; expected 1 or {n_features}."
        )
    return threshold.contiguous()


class GemmaScopeJumpReLUSAE(torch.nn.Module):
    """Minimal inference wrapper for Gemma Scope residual JumpReLU SAEs."""

    def __init__(self, params: Dict[str, torch.Tensor], device, dtype):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        def get_param(name: str, *aliases: str) -> torch.Tensor:
            for key in (name,) + aliases:
                if key in params:
                    return _as_tensor(params[key])
            raise KeyError(f"Missing SAE parameter {name!r}; keys={sorted(params)}")

        W_enc = get_param("W_enc")
        W_dec = get_param("W_dec")
        b_enc = get_param("b_enc").reshape(-1)
        b_dec = get_param("b_dec").reshape(-1)

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(
                f"Expected 2D W_enc/W_dec, got {W_enc.shape}, {W_dec.shape}"
            )

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])
        if W_dec.shape == (d_model, n_features):
            W_dec = W_dec.T.contiguous()
        if W_dec.shape != (n_features, d_model):
            raise ValueError(
                f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}"
            )
        if b_enc.numel() != n_features or b_dec.numel() != d_model:
            raise ValueError("Gemma Scope SAE bias shape mismatch.")

        threshold = _expand_threshold(
            get_param("threshold", "thresholds"), n_features
        )

        self.register_buffer("W_enc", W_enc.contiguous().to(self.device, self.dtype))
        self.register_buffer("W_dec", W_dec.contiguous().to(self.device, self.dtype))
        self.register_buffer("b_enc", b_enc.contiguous().to(self.device, self.dtype))
        self.register_buffer("b_dec", b_dec.contiguous().to(self.device, self.dtype))
        self.register_buffer("threshold", threshold.to(self.device, self.dtype))
        self.d_model = d_model
        self.n_features = n_features
        self.k = None

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        # Gemma Scope NPZ residual SAEs use the residual directly.
        return x.to(device=self.device, dtype=self.dtype)

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = self.process_sae_in(x)
        return x @ self.W_enc + self.b_enc

    def feature_fire_mask(self, pre: torch.Tensor) -> torch.Tensor:
        return pre > self.threshold

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(
            self.feature_fire_mask(pre), torch.relu(pre), torch.zeros_like(pre)
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class QwenBatchTopKSAE(torch.nn.Module):
    """Thresholded inference wrapper for Adam Karvonen's Qwen3 BatchTopK SAEs."""

    def __init__(self, state_dict, device, dtype, fallback_k=None):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if (
            isinstance(state_dict, dict)
            and "model" in state_dict
            and isinstance(state_dict["model"], dict)
        ):
            state_dict = state_dict["model"]

        def get_required(*names):
            for name in names:
                if name in state_dict and state_dict[name] is not None:
                    return _as_tensor(state_dict[name])
            raise KeyError(f"Could not find any of {names} in Qwen SAE checkpoint.")

        b_dec = get_required("b_dec", "decoder.bias").reshape(-1)
        b_enc = get_required("encoder.bias", "b_enc").reshape(-1)
        enc_w = get_required("encoder.weight", "W_enc")
        dec_w = get_required("decoder.weight", "W_dec")

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(f"Unexpected encoder weight shape {tuple(enc_w.shape)}")

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(f"Unexpected decoder weight shape {tuple(dec_w.shape)}")

        threshold = _expand_threshold(get_required("threshold"), n_features)
        k = state_dict.get("k", fallback_k)
        if torch.is_tensor(k):
            k = int(k.item())
        self.k = int(k) if k is not None else None

        self.register_buffer("W_enc", W_enc.to(self.device, self.dtype))
        self.register_buffer("W_dec", W_dec.to(self.device, self.dtype))
        self.register_buffer("b_enc", b_enc.to(self.device, self.dtype))
        self.register_buffer("b_dec", b_dec.to(self.device, self.dtype))
        self.register_buffer("threshold", threshold.to(self.device, self.dtype))
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return x - self.b_dec

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = self.process_sae_in(x)
        return x @ self.W_enc + self.b_enc

    def feature_fire_mask(self, pre: torch.Tensor) -> torch.Tensor:
        return pre > self.threshold

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(
            self.feature_fire_mask(pre), pre, torch.zeros_like(pre)
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class LlamaScopeResidualSAE(torch.nn.Module):
    """Direct inference wrapper for OpenMOSS Llama Scope residual SAEs."""

    def __init__(self, state_dict, hyperparams, device, dtype):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype
        self.hyperparams = dict(hyperparams)

        self.d_model = int(hyperparams["d_model"])
        self.n_features = int(hyperparams["d_sae"])
        self.k = int(hyperparams.get("top_k", 0) or 0) or None
        self.act_fn = str(hyperparams.get("act_fn", "jumprelu")).lower()
        self.sparsity_include_decoder_norm = bool(
            hyperparams.get("sparsity_include_decoder_norm", False)
        )
        self.apply_decoder_bias_to_pre_encoder = bool(
            hyperparams.get("apply_decoder_bias_to_pre_encoder", False)
        )

        def optional_tensor(candidate_keys, shape_options=None):
            for key in candidate_keys:
                if key in state_dict:
                    tensor = _as_tensor(state_dict[key]).cpu()
                    if shape_options is None or tuple(tensor.shape) in {
                        tuple(x) for x in shape_options
                    }:
                        return tensor
            for key, value in state_dict.items():
                if not torch.is_tensor(value):
                    continue
                tensor = value.detach().cpu()
                if shape_options is not None and tuple(tensor.shape) not in {
                    tuple(x) for x in shape_options
                }:
                    continue
                lowered = key.lower()
                if any(candidate.lower() in lowered for candidate in candidate_keys):
                    print(f"Using Llama Scope fuzzy key match: {key}")
                    return tensor
            return None

        W_enc = optional_tensor(
            ["W_E", "encoder.weight", "W_enc"],
            [(self.d_model, self.n_features), (self.n_features, self.d_model)],
        )
        W_dec = optional_tensor(
            ["W_D", "decoder.weight", "W_dec"],
            [(self.n_features, self.d_model), (self.d_model, self.n_features)],
        )
        if W_enc is None or W_dec is None:
            available = {
                key: tuple(value.shape)
                for key, value in state_dict.items()
                if torch.is_tensor(value)
            }
            raise KeyError(f"Could not locate Llama Scope weights. Available: {available}")
        if W_enc.shape == (self.n_features, self.d_model):
            W_enc = W_enc.T.contiguous()
        if W_dec.shape == (self.d_model, self.n_features):
            W_dec = W_dec.T.contiguous()

        b_enc = optional_tensor(
            ["b_E", "encoder.bias", "b_enc"], [(self.n_features,)]
        )
        b_dec = optional_tensor(
            ["b_D", "decoder.bias", "b_dec"], [(self.d_model,)]
        )
        if b_enc is None:
            b_enc = torch.zeros(self.n_features)
        if b_dec is None or not bool(hyperparams.get("use_decoder_bias", True)):
            b_dec = torch.zeros(self.d_model)

        log_threshold = optional_tensor(
            ["activation_function.log_jumprelu_threshold", "log_jumprelu_threshold"],
            [(self.n_features,), (1,)],
        )
        if log_threshold is not None:
            threshold = log_threshold.float().exp()
        else:
            threshold = optional_tensor(
                [
                    "activation_function.jumprelu_threshold",
                    "jumprelu_threshold",
                    "threshold",
                ],
                [(self.n_features,), (1,)],
            )
        if threshold is None:
            threshold = torch.tensor(float(hyperparams.get("jump_relu_threshold", 0.0)))
        threshold = _expand_threshold(threshold, self.n_features)

        # Match Llamascopium's inference-time parameter standardization for
        # dataset-wise activation normalization.
        self.norm_activation_standardized = False
        if (
            hyperparams.get("norm_activation") == "dataset-wise"
            and hyperparams.get("dataset_average_activation_norm") is not None
        ):
            average_norm = hyperparams["dataset_average_activation_norm"]
            hook_in = str(hyperparams.get("hook_point_in", "in"))
            hook_out = str(hyperparams.get("hook_point_out", hook_in))
            norm_in = float(average_norm.get(hook_in, average_norm.get("in")))
            norm_out = float(average_norm.get(hook_out, average_norm.get("out", norm_in)))
            input_norm_factor = math.sqrt(self.d_model) / norm_in
            output_norm_factor = math.sqrt(self.d_model) / norm_out
            b_enc = b_enc / input_norm_factor
            b_dec = b_dec / output_norm_factor
            W_dec = W_dec * (input_norm_factor / output_norm_factor)
            self.norm_activation_standardized = True

        decoder_norm = torch.linalg.vector_norm(W_dec.float(), dim=1).clamp_min(1e-8)

        self.register_buffer("W_enc", W_enc.to(self.device, self.dtype))
        self.register_buffer("W_dec", W_dec.to(self.device, self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).to(self.device, self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).to(self.device, self.dtype))
        self.register_buffer("threshold", threshold.to(self.device, self.dtype))
        self.register_buffer("decoder_norm", decoder_norm.to(self.device, self.dtype))

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        if self.apply_decoder_bias_to_pre_encoder:
            x = x - self.b_dec
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        pre = self.process_sae_in(x) @ self.W_enc + self.b_enc
        if self.sparsity_include_decoder_norm:
            pre = pre * self.decoder_norm
        return pre

    def feature_fire_mask(self, pre: torch.Tensor) -> torch.Tensor:
        if self.act_fn == "jumprelu":
            return pre > self.threshold
        if self.act_fn == "relu":
            return pre > 0
        if self.act_fn == "topk":
            return self.activation_fn(pre) != 0
        raise ValueError(f"Unsupported Llama Scope act_fn={self.act_fn!r}")

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        if self.act_fn == "jumprelu":
            acts = torch.where(pre > self.threshold, pre, torch.zeros_like(pre))
        elif self.act_fn == "relu":
            acts = torch.relu(pre)
        elif self.act_fn == "topk":
            if self.k is None or self.k <= 0:
                raise ValueError("Llama Scope TopK checkpoint has no positive k.")
            values, indices = torch.topk(
                pre, k=min(int(self.k), pre.shape[-1]), dim=-1
            )
            values = torch.relu(values)
            acts = torch.zeros_like(pre).scatter(-1, indices, values)
        else:
            raise ValueError(f"Unsupported Llama Scope act_fn={self.act_fn!r}")
        if self.sparsity_include_decoder_norm:
            acts = acts / self.decoder_norm
        return acts

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


def get_model_input_device(model) -> torch.device:
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int) -> torch.device:
    layer = model.get_submodule(f"model.layers.{int(layer_idx)}")
    try:
        return next(layer.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


def get_single_token_id(tokenizer, token: str) -> int:
    token_id = tokenizer.convert_tokens_to_ids(token)
    if token_id is not None and token_id != tokenizer.unk_token_id:
        return int(token_id)
    ids = tokenizer.encode(token, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Expected {token!r} to map to one token; got ids={ids}")
    return int(ids[0])


def load_configured_sae(device, dtype):
    print("Loading SAE:", SAE_REPO_ID, SAE_FILENAME)

    if SAE_BACKEND == "gemma_scope_npz":
        path = hf_hub_download(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            local_files_only=SAE_LOCAL_FILES_ONLY,
            token=False,
        )
        with np.load(path, allow_pickle=False) as params_np:
            params = {key: torch.from_numpy(params_np[key]) for key in params_np.files}
        return GemmaScopeJumpReLUSAE(params, device=device, dtype=dtype)

    if SAE_BACKEND == "qwen_batchtopk":
        path = hf_hub_download(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            local_files_only=SAE_LOCAL_FILES_ONLY,
            token=False,
        )
        try:
            state = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            state = torch.load(path, map_location="cpu")

        config = None
        if SAE_CONFIG_FILENAME is not None:
            config_path = hf_hub_download(
                repo_id=SAE_REPO_ID,
                filename=SAE_CONFIG_FILENAME,
                local_files_only=SAE_LOCAL_FILES_ONLY,
                token=False,
            )
            with open(config_path, "r", encoding="utf-8") as handle:
                config = json.load(handle)
            trainer_cfg = config.get("trainer", config)
            if int(trainer_cfg.get("layer", TARGET_LAYER)) != int(TARGET_LAYER):
                raise ValueError("Qwen SAE config layer does not match TARGET_LAYER.")
            if int(trainer_cfg.get("dict_size", EXPECTED_D_SAE)) != int(EXPECTED_D_SAE):
                raise ValueError("Qwen SAE config dictionary width is unexpected.")

        sae_obj = QwenBatchTopKSAE(
            state,
            device=device,
            dtype=dtype,
            fallback_k=SAE_TOPK_K,
        )
        return sae_obj

    if SAE_BACKEND == "llama_scope_direct":
        if SAE_HYPERPARAMS_FILENAME is None:
            raise ValueError("Llama Scope loading requires SAE_HYPERPARAMS_FILENAME.")
        try:
            from safetensors.torch import load_file as load_safetensors_file
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Install safetensors to load Llama Scope: pip install safetensors"
            ) from exc

        checkpoint_path = hf_hub_download(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            local_files_only=SAE_LOCAL_FILES_ONLY,
            token=False,
        )
        hyperparams_path = hf_hub_download(
            repo_id=SAE_REPO_ID,
            filename=SAE_HYPERPARAMS_FILENAME,
            local_files_only=SAE_LOCAL_FILES_ONLY,
            token=False,
        )
        with open(hyperparams_path, "r", encoding="utf-8") as handle:
            hyperparams = json.load(handle)
        state = load_safetensors_file(checkpoint_path, device="cpu")
        return LlamaScopeResidualSAE(
            state,
            hyperparams=hyperparams,
            device=device,
            dtype=dtype,
        )

    raise ValueError(f"Unknown SAE_BACKEND={SAE_BACKEND!r}")


tokenizer = load_tokenizer_direct()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = load_model_direct(MODEL_NAME)
MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)

sae = load_configured_sae(device=TARGET_LAYER_DEVICE, dtype=DTYPE)
sae.eval()

TURN_END_TOKEN = MODEL_TURN_END_TOKEN
TURN_END_TOKEN_ID = get_single_token_id(tokenizer, TURN_END_TOKEN)

_model_cfg = getattr(model.config, "text_config", model.config)
MODEL_HIDDEN_SIZE = int(getattr(_model_cfg, "hidden_size"))
MODEL_NUM_LAYERS = int(getattr(_model_cfg, "num_hidden_layers"))

if not (0 <= int(TARGET_LAYER) < MODEL_NUM_LAYERS):
    raise ValueError(
        f"TARGET_LAYER={TARGET_LAYER} is invalid for a {MODEL_NUM_LAYERS}-layer model."
    )
if int(sae.d_model) != MODEL_HIDDEN_SIZE:
    raise ValueError(
        f"SAE/model width mismatch: sae.d_model={sae.d_model}, "
        f"model hidden_size={MODEL_HIDDEN_SIZE}."
    )
if EXPECTED_MODEL_NUM_LAYERS is not None and MODEL_NUM_LAYERS != int(EXPECTED_MODEL_NUM_LAYERS):
    raise ValueError(
        f"Expected {EXPECTED_MODEL_NUM_LAYERS} model layers, found {MODEL_NUM_LAYERS}."
    )
if EXPECTED_D_MODEL is not None and MODEL_HIDDEN_SIZE != int(EXPECTED_D_MODEL):
    raise ValueError(
        f"Expected hidden size {EXPECTED_D_MODEL}, found {MODEL_HIDDEN_SIZE}."
    )
if EXPECTED_D_SAE is not None and int(sae.n_features) != int(EXPECTED_D_SAE):
    raise ValueError(
        f"Expected {EXPECTED_D_SAE} SAE features, found {sae.n_features}."
    )

print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)
print("MODEL_NUM_LAYERS:", MODEL_NUM_LAYERS)
print("MODEL_HIDDEN_SIZE:", MODEL_HIDDEN_SIZE)
print("SAE_BACKEND:", SAE_BACKEND)
print("SAE d_model / n_features / k:", sae.d_model, sae.n_features, getattr(sae, "k", None))
print("TURN_END_TOKEN_ID:", TURN_END_TOKEN_ID, repr(tokenizer.decode([TURN_END_TOKEN_ID])))
print("Direct loading and model/SAE preflight passed.")

## 2. Load AGNews and sample matched final examples

Demonstrations come from the AGNews training split. Example 8 articles come from the test split, with the same number sampled from every label.


In [ ]:
# ============================================================
# 2. Dataset loading and indexing
# ============================================================


def load_agnews_splits():
    attempts = []
    for dataset_name in ["ag_news", "fancyzhx/ag_news"]:
        try:
            ds = load_dataset(dataset_name)
            if "train" not in ds or "test" not in ds:
                raise ValueError(f"Dataset {dataset_name!r} lacks train/test splits")
            print("Loaded AGNews from:", dataset_name)
            return ds["train"], ds["test"], dataset_name
        except Exception as exc:
            attempts.append((dataset_name, repr(exc)))
    raise RuntimeError(f"Could not load AGNews. Attempts: {attempts}")


def index_by_label(dataset) -> Dict[int, List[int]]:
    by_label = {label: [] for label in ALL_LABELS}
    for idx, row in enumerate(dataset):
        label = int(row["label"])
        if label not in by_label:
            raise ValueError(f"Unexpected AGNews label {label}")
        by_label[label].append(idx)
    return by_label


def dataset_example(dataset, idx: int, split: str) -> Dict[str, Any]:
    row = dataset[int(idx)]
    return {
        "text": str(row["text"]),
        "label": int(row["label"]),
        "dataset_index": int(idx),
        "split": str(split),
    }


train_data_agnews, test_data_agnews, AGNEWS_DATASET_NAME = load_agnews_splits()
train_by_label = index_by_label(train_data_agnews)
test_by_label = index_by_label(test_data_agnews)

print("Train counts:", {LABEL_TO_WORD[l]: len(train_by_label[l]) for l in ALL_LABELS})
print("Test counts:", {LABEL_TO_WORD[l]: len(test_by_label[l]) for l in ALL_LABELS})


def sample_final_examples(per_label: int, seed: int) -> Dict[int, List[Dict[str, Any]]]:
    rng = random.Random(seed)
    out = {}
    for label in ALL_LABELS:
        pool = test_by_label[label]
        if len(pool) < per_label:
            raise ValueError(f"Not enough test examples for label {label}: {len(pool)} < {per_label}")
        chosen = rng.sample(pool, per_label)
        out[label] = [dataset_example(test_data_agnews, idx, "test") for idx in chosen]
    return out


final_examples_by_label = sample_final_examples(NUM_FINAL_EXAMPLES_PER_LABEL, SEED)
print("Sampled final Example-8 articles per label:", {
    LABEL_TO_WORD[label]: len(rows) for label, rows in final_examples_by_label.items()
})


## 3. Construct the paired 8-example prompts

For a retained-label final example, the original prompt contains exactly two examples per label. The target prompt preserves the same Example 8 and all five retained-label context examples, then replaces only the two removed-label context examples at the same positions. One replacement is drawn from each of two retained labels, producing a balanced `3/3/2` target distribution.


In [ ]:
# ============================================================
# 3. Prompt rendering and tightly paired prompt construction
# ============================================================

SPAN_START_SENTINEL = "ZXQ_AGNEWS_SPAN_START_7f3c9b0a"
SPAN_END_SENTINEL = "ZXQ_AGNEWS_SPAN_END_7f3c9b0a"


def clean_agnews_text(text: str, max_chars: Optional[int] = MAX_ARTICLE_CHARS) -> str:
    text = str(text).replace("\n", " ").replace("\t", " ").strip()
    text = re.sub(r"\s+", " ", text)
    if max_chars is not None and len(text) > max_chars:
        cut = text[:max_chars].rstrip()
        last_space = cut.rfind(" ")
        if last_space >= int(0.60 * max_chars):
            cut = cut[:last_space].rstrip()
        text = cut + "..."
    return text


def format_label_list(label_ids: Sequence[int]) -> str:
    names = [LABEL_TO_WORD[int(label)] for label in label_ids]
    if len(names) == 1:
        return names[0]
    if len(names) == 2:
        return f"{names[0]} and {names[1]}"
    return ", ".join(names[:-1]) + f", and {names[-1]}"


def instruction_for_allowed_labels(label_ids: Sequence[int]) -> str:
    # The first sentence is deliberately condition-specific, as requested.
    first_sentence = f"The possible news topic labels are {format_label_list(label_ids)}."
    second_sentence = (
        "Each of the following eight examples contains an article followed by its correct "
        "topic label; read the examples carefully."
    )
    return first_sentence + " " + second_sentence


def format_labeled_example(example: Dict[str, Any], idx: int) -> str:
    text = clean_agnews_text(example["text"])
    label_name = LABEL_TO_WORD[int(example["label"])]
    return (
        f"Example {idx}\n"
        f"Article:\n{text}\n"
        f"Topic:\n{label_name}"
    )


def apply_model_chat_template(
    messages: List[Dict[str, str]],
    *,
    add_generation_prompt: bool = False,
) -> str:
    kwargs = {
        "tokenize": False,
        "add_generation_prompt": bool(add_generation_prompt),
    }
    if MODEL_SUPPORTS_ENABLE_THINKING:
        kwargs["enable_thinking"] = False
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop("enable_thinking", None)
        return tokenizer.apply_chat_template(messages, **kwargs)


def render_prompt_with_final_span(
    examples_8: Sequence[Dict[str, Any]],
    allowed_labels: Sequence[int],
) -> Tuple[str, Tuple[int, int], str]:
    """
    Render one user-turn chat-template prompt and return:
      1. cleaned rendered prompt,
      2. character span of Example 8 content, excluding the literal `Example 8` header,
      3. the condition-specific first instruction sentence.
    """
    if len(examples_8) != N_EXAMPLES_PER_PROMPT:
        raise ValueError(f"Expected {N_EXAMPLES_PER_PROMPT} examples, got {len(examples_8)}")

    instruction = instruction_for_allowed_labels(allowed_labels)
    first_sentence = instruction.split(". ", 1)[0] + "."
    blocks = [format_labeled_example(example, idx + 1) for idx, example in enumerate(examples_8)]

    final_block = blocks[-1]
    expected_header = "Example 8\n"
    if not final_block.startswith(expected_header):
        raise ValueError(f"Final block does not start with {expected_header!r}")

    final_content = final_block[len(expected_header):]
    marked_final_block = expected_header + SPAN_START_SENTINEL + final_content + SPAN_END_SENTINEL
    user_content_marked = instruction + "\n\n" + "\n\n".join(blocks[:-1] + [marked_final_block])

    rendered_marked = apply_model_chat_template([
        {"role": "user", "content": user_content_marked}
    ])

    start_pos = rendered_marked.find(SPAN_START_SENTINEL)
    end_pos = rendered_marked.find(SPAN_END_SENTINEL)
    if start_pos < 0 or end_pos < 0 or end_pos <= start_pos:
        raise ValueError(
            "Could not recover final Example-8 sentinels after chat-template rendering."
        )

    rendered_final_content = rendered_marked[
        start_pos + len(SPAN_START_SENTINEL):end_pos
    ]
    if rendered_final_content != final_content:
        raise ValueError("Chat template changed the marked final Example-8 content.")

    prompt = (
        rendered_marked[:start_pos]
        + final_content
        + rendered_marked[end_pos + len(SPAN_END_SENTINEL):]
    ).rstrip()
    final_char_span = (start_pos, start_pos + len(final_content))

    actual = prompt[final_char_span[0]:final_char_span[1]]
    if actual != final_content:
        raise ValueError("Final Example-8 span mapping failed.")

    return prompt, final_char_span, first_sentence


def sample_indices_excluding(
    pool: Sequence[int],
    k: int,
    rng: random.Random,
    exclude: Optional[set] = None,
) -> List[int]:
    exclude = set(exclude or set())
    chosen = []
    chosen_set = set()
    max_attempts = max(1000, 100 * k)
    attempts = 0
    while len(chosen) < k and attempts < max_attempts:
        idx = int(rng.choice(pool))
        attempts += 1
        if idx in exclude or idx in chosen_set:
            continue
        chosen.append(idx)
        chosen_set.add(idx)
    if len(chosen) != k:
        candidates = [int(idx) for idx in pool if int(idx) not in exclude and int(idx) not in chosen_set]
        need = k - len(chosen)
        if len(candidates) < need:
            raise ValueError(f"Could not sample {k} unique indices after exclusions")
        chosen.extend(rng.sample(candidates, need))
    return chosen


def label_counts(examples: Sequence[Dict[str, Any]]) -> Dict[int, int]:
    return {
        label: sum(int(example["label"]) == label for example in examples)
        for label in ALL_LABELS
    }


def example_key(example: Dict[str, Any]) -> str:
    return f"{example['split']}:{int(example['dataset_index'])}"


def final_text_hash(example: Dict[str, Any]) -> str:
    payload = clean_agnews_text(example["text"]).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()


def build_original_examples(final_example: Dict[str, Any], pair_id: int):
    final_label = int(final_example["label"])
    desired_context_counts = {label: ORIGINAL_EXAMPLES_PER_LABEL for label in ALL_LABELS}
    desired_context_counts[final_label] -= 1

    rng = random.Random(SEED + 10_007 * (pair_id + 1))
    context = []
    used_train_indices = set()

    for label in ALL_LABELS:
        count = desired_context_counts[label]
        selected = sample_indices_excluding(
            train_by_label[label],
            count,
            rng,
            exclude=used_train_indices,
        )
        used_train_indices.update(selected)
        context.extend(dataset_example(train_data_agnews, idx, "train") for idx in selected)

    rng.shuffle(context)
    examples_8 = context + [dict(final_example)]

    counts = label_counts(examples_8)
    expected = {label: ORIGINAL_EXAMPLES_PER_LABEL for label in ALL_LABELS}
    if counts != expected:
        raise AssertionError(f"Original label counts {counts} != expected {expected}")

    return examples_8


def build_target_examples_from_original(
    original_examples_8: Sequence[Dict[str, Any]],
    pair_id: int,
):
    final_example = dict(original_examples_8[-1])
    final_label = int(final_example["label"])
    if final_label == target_label:
        raise ValueError("Target prompts cannot have Example 8 from the removed label")

    original_context = [dict(example) for example in original_examples_8[:-1]]
    removed_positions = [
        pos for pos, example in enumerate(original_context)
        if int(example["label"]) == target_label
    ]
    if len(removed_positions) != 2:
        raise AssertionError(
            f"Expected exactly two removed-label context examples; got {removed_positions}"
        )

    rng = random.Random(SEED + 30_011 * (pair_id + 1))

    # Original totals are 2/2/2 over retained labels. Add one replacement to two
    # distinct retained labels, yielding a balanced 3/3/2 target distribution.
    replacement_labels = list(kept_labels)
    rng.shuffle(replacement_labels)
    replacement_labels = replacement_labels[:2]

    used_train_indices = {
        int(example["dataset_index"])
        for example in original_context
        if example["split"] == "train"
    }

    target_context = [dict(example) for example in original_context]
    replacement_records = []

    for position, replacement_label in zip(removed_positions, replacement_labels):
        replacement_idx = sample_indices_excluding(
            train_by_label[replacement_label],
            1,
            rng,
            exclude=used_train_indices,
        )[0]
        used_train_indices.add(replacement_idx)
        replacement_example = dataset_example(train_data_agnews, replacement_idx, "train")
        target_context[position] = replacement_example
        replacement_records.append({
            "position": int(position + 1),
            "replacement_label": int(replacement_label),
            "replacement_train_index": int(replacement_idx),
        })

    examples_8 = target_context + [final_example]
    counts = label_counts(examples_8)

    if counts[target_label] != 0:
        raise AssertionError(f"Removed label still appears in target prompt: counts={counts}")
    retained_counts = [counts[label] for label in kept_labels]
    if sorted(retained_counts) != [2, 3, 3]:
        raise AssertionError(f"Expected retained target counts [2,3,3], got {counts}")

    shared_positions = [
        pos for pos in range(7)
        if example_key(original_context[pos]) == example_key(target_context[pos])
    ]
    if len(shared_positions) != 5:
        raise AssertionError(f"Expected five shared context positions; got {shared_positions}")

    return examples_8, replacement_records, shared_positions


def build_paired_prompt_sets():
    original_prompts = []
    original_meta = []
    target_prompts = []
    target_meta = []

    pair_id = 0
    for final_label in ALL_LABELS:
        for within_label_idx, final_example in enumerate(final_examples_by_label[final_label]):
            original_examples_8 = build_original_examples(final_example, pair_id)
            original_prompt, original_span, original_first_sentence = render_prompt_with_final_span(
                original_examples_8,
                allowed_labels=ALL_LABELS,
            )

            original_prompts.append(original_prompt)
            original_meta.append({
                "pair_id": int(pair_id),
                "within_label_idx": int(within_label_idx),
                "final_label": int(final_label),
                "final_label_name": LABEL_TO_WORD[int(final_label)],
                "final_example_key": example_key(final_example),
                "final_text_hash": final_text_hash(final_example),
                "condition": "original_four_labels",
                "allowed_labels": list(ALL_LABELS),
                "instruction_first_sentence": original_first_sentence,
                "label_counts": label_counts(original_examples_8),
                "example_keys": [example_key(example) for example in original_examples_8],
                "example_labels": [int(example["label"]) for example in original_examples_8],
                "final_char_span": tuple(map(int, original_span)),
            })

            if final_label != target_label:
                target_examples_8, replacements, shared_positions = build_target_examples_from_original(
                    original_examples_8,
                    pair_id,
                )
                target_prompt, target_span, target_first_sentence = render_prompt_with_final_span(
                    target_examples_8,
                    allowed_labels=kept_labels,
                )

                target_prompts.append(target_prompt)
                target_meta.append({
                    "pair_id": int(pair_id),
                    "within_label_idx": int(within_label_idx),
                    "final_label": int(final_label),
                    "final_label_name": LABEL_TO_WORD[int(final_label)],
                    "final_example_key": example_key(final_example),
                    "final_text_hash": final_text_hash(final_example),
                    "condition": f"target_without_{LABEL_TO_WORD[target_label].lower()}",
                    "removed_label": int(target_label),
                    "removed_label_name": LABEL_TO_WORD[int(target_label)],
                    "allowed_labels": list(kept_labels),
                    "instruction_first_sentence": target_first_sentence,
                    "label_counts": label_counts(target_examples_8),
                    "example_keys": [example_key(example) for example in target_examples_8],
                    "example_labels": [int(example["label"]) for example in target_examples_8],
                    "replacement_records": replacements,
                    "shared_context_positions_1indexed": [int(pos + 1) for pos in shared_positions],
                    "final_char_span": tuple(map(int, target_span)),
                })

            pair_id += 1

    return original_prompts, original_meta, target_prompts, target_meta


original_prompts, original_meta, target_prompts, target_meta = build_paired_prompt_sets()

print("Original prompts:", len(original_prompts))
print("Target prompts:", len(target_prompts))
print("Original final-label counts:")
display(pd.DataFrame(original_meta).groupby("final_label_name").size().rename("n"))
print("Target final-label counts:")
display(pd.DataFrame(target_meta).groupby("final_label_name").size().rename("n"))


In [ ]:
# ============================================================
# 4. Pairing and prompt-composition sanity checks
# ============================================================

original_by_pair_id = {int(meta["pair_id"]): meta for meta in original_meta}

for target_item in target_meta:
    pair_id = int(target_item["pair_id"])
    original_item = original_by_pair_id[pair_id]

    assert original_item["final_example_key"] == target_item["final_example_key"]
    assert original_item["final_text_hash"] == target_item["final_text_hash"]
    assert original_item["final_label"] == target_item["final_label"]
    assert original_item["example_keys"][-1] == target_item["example_keys"][-1]

    original_counts = original_item["label_counts"]
    target_counts = target_item["label_counts"]

    assert all(int(original_counts[label]) == 2 for label in ALL_LABELS)
    assert int(target_counts[target_label]) == 0
    assert sorted(int(target_counts[label]) for label in kept_labels) == [2, 3, 3]
    assert len(target_item["shared_context_positions_1indexed"]) == 5

expected_original_first_sentence = (
    f"The possible news topic labels are {format_label_list(ALL_LABELS)}."
)
expected_target_first_sentence = (
    f"The possible news topic labels are {format_label_list(kept_labels)}."
)

assert all(
    item["instruction_first_sentence"] == expected_original_first_sentence
    for item in original_meta
)
assert all(
    item["instruction_first_sentence"] == expected_target_first_sentence
    for item in target_meta
)

print("PASS: Example 8 is exactly matched for every original/target pair.")
print("PASS: five of seven context positions are shared; two removed-label contexts are replaced.")
print("PASS: original label counts are 2/2/2/2.")
print("PASS: target label counts are 3/3/2 over retained labels and zero for the removed label.")
print("Original first sentence:", expected_original_first_sentence)
print("Target first sentence:  ", expected_target_first_sentence)

print("\nOriginal prompt preview:\n")
print(original_prompts[0][:2500])
print("\nOriginal prompt tail:\n", repr(original_prompts[0][-700:]))

print("\nTarget prompt preview:\n")
print(target_prompts[0][:2500])
print("\nTarget prompt tail:\n", repr(target_prompts[0][-700:]))


## 4. Locate the final Example-8 score tokens

The marked character span excludes the literal `Example 8` header and includes `Article:`, the article text, `Topic:`, and the gold label. The span is extended through the model's following `<|im_end|>` token so that `LAST_K_TOKENS` can use explicit offsets from the span end: `-1` is `<|im_end|>`, `-2` is the immediately preceding token, and so forth. With the default `LAST_K_TOKENS = [-2, -3, -4]`, the notebook averages the three tokens immediately before `<|im_end|>` and excludes the turn-ending token itself. The debugging cell prints the exact selected tokens for both conditions.


In [ ]:
# ============================================================
# 5. Character-to-token span helpers and exact-token debugging
# ============================================================


def _to_id_list(input_ids_1d) -> List[int]:
    if torch.is_tensor(input_ids_1d):
        return [int(value) for value in input_ids_1d.detach().cpu().tolist()]
    return [int(value) for value in input_ids_1d]


def char_span_to_token_span(
    offset_mapping: Sequence[Tuple[int, int]],
    char_start: int,
    char_end: int,
) -> Tuple[int, int]:
    token_ids = []
    for token_idx, (start, end) in enumerate(offset_mapping):
        if start == 0 and end == 0:
            continue
        if end > char_start and start < char_end:
            token_ids.append(token_idx)
    if not token_ids:
        raise ValueError(f"Could not map char span [{char_start}, {char_end}) to tokens")
    return token_ids[0], token_ids[-1] + 1


def extend_span_to_next_turn_end(
    input_ids_1d,
    tok_start: int,
    tok_end: int,
    max_search_tokens: int = 64,
) -> Tuple[int, int]:
    ids = _to_id_list(input_ids_1d)
    if tok_end > tok_start and ids[tok_end - 1] == TURN_END_TOKEN_ID:
        return tok_start, tok_end

    search_end = min(len(ids), tok_end + max_search_tokens)
    for token_idx in range(tok_end, search_end):
        if ids[token_idx] == TURN_END_TOKEN_ID:
            return tok_start, token_idx + 1

    nearby = [
        (idx, ids[idx], repr(tokenizer.decode([ids[idx]])))
        for idx in range(max(0, tok_end - 8), search_end)
    ]
    raise RuntimeError(
        f"Could not find the next {TURN_END_TOKEN!r}; "
        f"tok_start={tok_start}, tok_end={tok_end}, nearby={nearby}"
    )


def get_final_example_token_span(
    offset_mapping,
    final_char_span,
    input_ids_1d,
    include_turn_end: bool = INCLUDE_FINAL_TURN_END_IN_SCORE_SPAN,
) -> Tuple[int, int]:
    char_start, char_end = map(int, final_char_span)
    tok_start, tok_end = char_span_to_token_span(
        offset_mapping,
        char_start,
        char_end,
    )
    if include_turn_end:
        tok_start, tok_end = extend_span_to_next_turn_end(
            input_ids_1d,
            tok_start,
            tok_end,
        )
    return tok_start, tok_end


def normalize_score_offsets_from_end(offsets: Sequence[int]) -> List[int]:
    """
    Normalize explicit negative offsets counted from the exclusive span end.

    For a span [tok_start, tok_end):
      -1 -> tok_end - 1 (the final token; <|im_end|> when extension is enabled)
      -2 -> tok_end - 2
      -3 -> tok_end - 3
      ...
    """
    if not isinstance(offsets, (list, tuple)) or len(offsets) == 0:
        raise ValueError(
            "Score offsets must be a non-empty list/tuple of negative integers, "
            "e.g. [-2, -3, -4]."
        )

    normalized = []
    for offset in offsets:
        if isinstance(offset, bool) or not isinstance(offset, (int, np.integer)):
            raise TypeError(
                "Every score offset must be a negative integer; "
                f"got {offset!r}."
            )
        offset = int(offset)
        if offset >= 0:
            raise ValueError(
                "Score offsets are counted backward from the exclusive span end, "
                f"so every offset must be negative; got {offset}."
            )
        normalized.append(offset)

    if len(set(normalized)) != len(normalized):
        raise ValueError(f"Duplicate score offsets are not allowed: {normalized}.")

    return normalized


def get_score_indices_from_end_offsets(
    tok_start: int,
    tok_end: int,
    offsets: Sequence[int],
) -> List[int]:
    """Map explicit end-relative offsets to valid token indices, left to right."""
    if tok_end <= tok_start:
        raise ValueError(f"Invalid token span [{tok_start}, {tok_end})")

    normalized = normalize_score_offsets_from_end(offsets)
    score_indices = []

    for offset in normalized:
        token_idx = tok_end + offset
        if token_idx < tok_start or token_idx >= tok_end:
            span_length = tok_end - tok_start
            raise IndexError(
                f"Offset {offset} falls outside token span [{tok_start}, {tok_end}) "
                f"of length {span_length}. Configured offsets: {normalized}."
            )
        score_indices.append(token_idx)

    # Mean/any reductions are order-invariant, but left-to-right ordering makes
    # slicing and token debugging easier to interpret.
    return sorted(score_indices)


def debug_final_scoring_tokens(prompt: str, meta_item: Dict[str, Any]) -> None:
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    input_ids = encoded["input_ids"][0]
    offsets = encoded["offset_mapping"][0].tolist()

    tok_start, tok_end = get_final_example_token_span(
        offsets,
        meta_item["final_char_span"],
        input_ids,
    )
    normalized_offsets = normalize_score_offsets_from_end(LAST_K_TOKENS)
    offset_to_index = [(offset, tok_end + offset) for offset in normalized_offsets]
    score_indices = get_score_indices_from_end_offsets(
        tok_start,
        tok_end,
        normalized_offsets,
    )

    print("=" * 100)
    print("Condition:", meta_item["condition"])
    print("Final label:", meta_item["final_label_name"])
    print("Final token span:", (tok_start, tok_end))
    print("Configured offsets from span end:", normalized_offsets)
    print("Offset -> absolute token index:", offset_to_index)
    print("Score indices in left-to-right order:", score_indices)
    print("Prompt tail:", repr(prompt[-500:]))
    print("\nLast tokens in final Example-8 span:")

    for idx in range(max(tok_start, tok_end - 12), tok_end):
        token_id = int(input_ids[idx])
        start, end = offsets[idx]
        text_piece = prompt[start:end] if end > start else ""
        marker = "  <-- SCORED" if idx in score_indices else ""
        print(
            f"idx={idx:>5} id={token_id:<8} "
            f"token={repr(tokenizer.decode([token_id])):<24} "
            f"offset={(start, end)} text={repr(text_piece)}{marker}"
        )

    print("\nSelected tokens by configured offset:")
    for offset, idx in offset_to_index:
        token_id = int(input_ids[idx])
        start, end = offsets[idx]
        text_piece = prompt[start:end] if end > start else ""
        print(
            f"offset={offset:>4} -> idx={idx:>5} id={token_id:<8} "
            f"token={repr(tokenizer.decode([token_id])):<24} "
            f"offset_mapping={(start, end)} text={repr(text_piece)}"
        )

    scored_ids = [int(input_ids[idx]) for idx in score_indices]
    print(
        "\nDecoded selected tokens in left-to-right order:",
        repr(tokenizer.decode(scored_ids)),
    )

    if INCLUDE_FINAL_TURN_END_IN_SCORE_SPAN:
        final_span_token_id = int(input_ids[tok_end - 1])
        assert final_span_token_id == TURN_END_TOKEN_ID, (
            f"Expected extended span to end in {TURN_END_TOKEN!r}, but found "
            f"{repr(tokenizer.decode([final_span_token_id]))}."
        )
        if -1 in normalized_offsets:
            assert tok_end - 1 in score_indices
            print(f"PASS: configured offsets include {TURN_END_TOKEN!r} via -1.")
        else:
            assert tok_end - 1 not in score_indices
            print(f"PASS: configured offsets exclude {TURN_END_TOKEN!r}.")

    print("=" * 100)


# Lightweight selector sanity check, independent of the tokenizer/model.
assert get_score_indices_from_end_offsets(10, 20, [-2, -3, -4]) == [16, 17, 18]
assert 19 not in get_score_indices_from_end_offsets(10, 20, [-2, -3, -4])
print("PASS: [-2, -3, -4] selects the three tokens before the final span token.")

# Show one paired final example and verify the exact selected tokens.
debug_final_scoring_tokens(original_prompts[0], original_meta[0])
debug_final_scoring_tokens(target_prompts[0], target_meta[0])


## 5. Extract layer-20 SAE feature values and firing indicators

For each prompt, a forward hook captures the output of transformer layer 20. Only the selected final score-token residual vectors are passed through the SAE encoder, avoiding the memory cost of encoding every prompt token into all 16,384 features.

- `activation_type = "pre"`: average the selected-token SAE preactivations.
- `activation_type = "post"`: average the selected-token post-gate sparse activations.
- Independently of `activation_type`, the notebook records whether each feature is active on at least one selected score token. These instance-level firing indicators define the frequency filter.


In [7]:
# ============================================================
# 6. Efficient residual capture and SAE score/firing extraction
# ============================================================


@torch.no_grad()
def get_layer_output(model, input_ids, attention_mask, layer_idx: int) -> torch.Tensor:
    captured = {}

    def hook_fn(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["hidden"] = hidden.detach()
        return output

    block = model.get_submodule(f"model.layers.{layer_idx}")
    handle = block.register_forward_hook(hook_fn)
    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        handle.remove()

    if "hidden" not in captured:
        raise RuntimeError(f"Failed to capture model.layers.{layer_idx}")
    return captured["hidden"]


@torch.no_grad()
def extract_example_level_scores(
    prompts: Sequence[str],
    meta: Sequence[Dict[str, Any]],
    batch_size: int = BATCH_SIZE,
    activation_kind: Optional[str] = None,
    return_both: bool = False,
    verbose: bool = True,
    progress_label: str = "batch",
):
    """
    Extract Example-8 SAE values at the configured explicit score-token offsets.

    Returns
    -------
    If return_both=False:
        scores: [n_examples, n_features], selected by activation_kind;
        fired:  [n_examples, n_features], boolean feature-firing indicators.

    If return_both=True:
        pre_scores:  [n_examples, n_features], mean encoder preactivation;
        post_scores: [n_examples, n_features], mean post-threshold activation;
        fired:       [n_examples, n_features], boolean firing indicators.

    An example-level feature is marked as fired when it is active under its configured SAE gate on at least one token selected by LAST_K_TOKENS.
    """
    if len(prompts) != len(meta):
        raise ValueError(f"prompts length {len(prompts)} != meta length {len(meta)}")

    if activation_kind is None:
        activation_kind = activation_type
    activation_kind = str(activation_kind).strip().lower()
    if activation_kind not in {"pre", "post"}:
        raise ValueError(
            f"activation_kind must be 'pre' or 'post'; got {activation_kind!r}."
        )

    all_pre_scores = []
    all_post_scores = []
    all_fired = []
    n_batches = math.ceil(len(prompts) / batch_size)

    for batch_number, batch_start in enumerate(range(0, len(prompts), batch_size), start=1):
        batch_prompts = list(prompts[batch_start:batch_start + batch_size])
        batch_meta = list(meta[batch_start:batch_start + batch_size])

        encoded = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,
            add_special_tokens=False,
            return_offsets_mapping=True,
        )

        input_ids = encoded["input_ids"].to(MODEL_INPUT_DEVICE)
        attention_mask = encoded["attention_mask"].to(MODEL_INPUT_DEVICE)
        offset_mapping = encoded["offset_mapping"]

        score_indices_by_example = []
        for batch_idx, meta_item in enumerate(batch_meta):
            offsets = offset_mapping[batch_idx].tolist()
            input_ids_cpu = input_ids[batch_idx].detach().cpu()
            tok_start, tok_end = get_final_example_token_span(
                offsets,
                meta_item["final_char_span"],
                input_ids_cpu,
            )
            score_indices_by_example.append(
                get_score_indices_from_end_offsets(tok_start, tok_end, LAST_K_TOKENS)
            )

        hidden = get_layer_output(
            model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            layer_idx=TARGET_LAYER,
        )

        for batch_idx, score_indices in enumerate(score_indices_by_example):
            selected_hidden = hidden[batch_idx, score_indices, :]
            preactivation = sae.encode_pre(selected_hidden)
            fire_mask = sae.feature_fire_mask(preactivation)
            postactivation = sae.activation_fn(preactivation)

            pre_score = preactivation.detach().float().mean(dim=0).cpu()
            post_score = postactivation.detach().float().mean(dim=0).cpu()
            pre_score = torch.nan_to_num(pre_score, nan=0.0, posinf=0.0, neginf=0.0)
            post_score = torch.nan_to_num(post_score, nan=0.0, posinf=0.0, neginf=0.0)

            fired = fire_mask.detach().any(dim=0).cpu().bool()

            all_pre_scores.append(pre_score)
            all_post_scores.append(post_score)
            all_fired.append(fired)

        if verbose and (
            batch_number == 1
            or batch_number == n_batches
            or batch_number % PRINT_EVERY_BATCHES == 0
        ):
            print(
                f"{progress_label} {batch_number}/{n_batches} | "
                f"examples {batch_start}:{batch_start + len(batch_prompts)}"
            )

        del hidden
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pre_scores = torch.stack(all_pre_scores, dim=0).float()
    post_scores = torch.stack(all_post_scores, dim=0).float()
    fired = torch.stack(all_fired, dim=0).bool()

    for name, values in {
        "pre_scores": pre_scores,
        "post_scores": post_scores,
    }.items():
        if not torch.isfinite(values).all():
            raise RuntimeError(f"Non-finite values remain in {name}")

    if pre_scores.shape != post_scores.shape or pre_scores.shape != fired.shape:
        raise RuntimeError(
            "Pre/post/firing shape mismatch: "
            f"{tuple(pre_scores.shape)}, {tuple(post_scores.shape)}, {tuple(fired.shape)}"
        )

    if return_both:
        return pre_scores, post_scores, fired

    selected_scores = pre_scores if activation_kind == "pre" else post_scores
    return selected_scores, fired


In [ ]:
# ============================================================
# 7. Run original and target extraction
# ============================================================

# Retain both original pre- and post-threshold values from the same forward
# passes. The main AFR analysis still follows the global activation_type.
example_level_preactivation_original, example_level_postactivation_original, example_level_fired_original = (
    extract_example_level_scores(
        original_prompts,
        original_meta,
        batch_size=BATCH_SIZE,
        activation_kind=activation_type,
        return_both=True,
    )
)

example_level_score_original = (
    example_level_preactivation_original
    if activation_type == "pre"
    else example_level_postactivation_original
)

# The target condition only needs the globally selected activation type for the
# activation-redistribution analysis.
example_level_score_target, example_level_fired_target = extract_example_level_scores(
    target_prompts,
    target_meta,
    batch_size=BATCH_SIZE,
    activation_kind=activation_type,
    return_both=False,
)

print("activation_type:", activation_type, f"({ACTIVATION_VALUE_NAME})")
print("Original selected-score shape:", tuple(example_level_score_original.shape))
print("Original preactivation shape: ", tuple(example_level_preactivation_original.shape))
print("Original postactivation shape:", tuple(example_level_postactivation_original.shape))
print("Target score shape:           ", tuple(example_level_score_target.shape))
print("Original firing shape:        ", tuple(example_level_fired_original.shape))
print("Target firing shape:          ", tuple(example_level_fired_target.shape))
print("Number of SAE features:", sae.n_features)

expected_original_shape = (len(original_prompts), sae.n_features)
expected_target_shape = (len(target_prompts), sae.n_features)

assert example_level_score_original.shape == expected_original_shape
assert example_level_preactivation_original.shape == expected_original_shape
assert example_level_postactivation_original.shape == expected_original_shape
assert example_level_score_target.shape == expected_target_shape
assert example_level_fired_original.shape == expected_original_shape
assert example_level_fired_target.shape == expected_target_shape
assert example_level_fired_original.dtype == torch.bool
assert example_level_fired_target.dtype == torch.bool


## 6. Align paired rows and identify removed-label-selective features

Feature selectivity is computed only from the original four-label condition using the quantity chosen by `activation_type`. A feature is assigned to the removed label when that label has the largest original average value; its selectivity score is the margin over the strongest other label.

The frequency filter is separate from this selectivity ranking. It uses actual feature-firing events under the configured SAE gate on original-condition final examples whose label is `target_label`.


In [ ]:
# ============================================================
# 8. Align original and target rows by the exact final Example-8 pair ID
# ============================================================

original_row_by_pair_id = {
    int(meta_item["pair_id"]): row_idx
    for row_idx, meta_item in enumerate(original_meta)
}

aligned_original_rows = []
for target_item in target_meta:
    pair_id = int(target_item["pair_id"])
    if pair_id not in original_row_by_pair_id:
        raise ValueError(f"Target pair_id={pair_id} missing from original set")
    original_row = original_row_by_pair_id[pair_id]
    original_item = original_meta[original_row]

    assert original_item["final_example_key"] == target_item["final_example_key"]
    assert original_item["final_text_hash"] == target_item["final_text_hash"]
    assert original_item["final_label"] == target_item["final_label"]
    aligned_original_rows.append(original_row)

aligned_original_rows = torch.tensor(aligned_original_rows, dtype=torch.long)
example_level_score_original_aligned = example_level_score_original[aligned_original_rows]
paired_feature_shift = example_level_score_target - example_level_score_original_aligned

assert example_level_score_original_aligned.shape == example_level_score_target.shape

pair_df = pd.DataFrame(target_meta)[[
    "pair_id",
    "final_label",
    "final_label_name",
    "final_example_key",
]].copy()
pair_df["original_row"] = aligned_original_rows.numpy()
pair_df["target_row"] = np.arange(len(pair_df))

display(pair_df.groupby("final_label_name").size().rename("paired_n"))
print("Pooled paired rows:", len(pair_df))


In [ ]:
# ============================================================
# 9. Label-wise means and JumpReLU firing frequencies
# ============================================================


def get_rows_for_label(
    meta: Sequence[Dict[str, Any]],
    label: int,
) -> List[int]:
    return [
        idx for idx, meta_item in enumerate(meta)
        if int(meta_item["final_label"]) == int(label)
    ]


def compute_label_wise_mean(
    score_matrix: torch.Tensor,
    meta: Sequence[Dict[str, Any]],
    label_ids: Sequence[int],
) -> Tuple[torch.Tensor, Dict[int, int]]:
    rows = []
    counts = {}
    for label in label_ids:
        indices = get_rows_for_label(meta, label)
        if not indices:
            raise ValueError(f"No final examples found for label {label}")
        index_tensor = torch.tensor(indices, dtype=torch.long)
        rows.append(score_matrix[index_tensor].mean(dim=0))
        counts[int(label)] = len(indices)
    return torch.stack(rows, dim=0), counts


def compute_label_wise_firing_frequency(
    fired_matrix: torch.Tensor,
    meta: Sequence[Dict[str, Any]],
    label_ids: Sequence[int],
) -> Tuple[torch.Tensor, Dict[int, int]]:
    if fired_matrix.dtype != torch.bool:
        raise TypeError(f"fired_matrix must be boolean; got {fired_matrix.dtype}")

    rows = []
    counts = {}
    for label in label_ids:
        indices = get_rows_for_label(meta, label)
        if not indices:
            raise ValueError(f"No final examples found for label {label}")
        index_tensor = torch.tensor(indices, dtype=torch.long)
        rows.append(fired_matrix[index_tensor].float().mean(dim=0))
        counts[int(label)] = len(indices)
    return torch.stack(rows, dim=0), counts


# Keep both original-condition preactivation and postactivation means. The main
# activation analysis follows activation_type, whereas causal feature eligibility
# is deliberately defined using preactivation means, independent of activation_type.
original_preactivation_label_mean, original_label_counts = compute_label_wise_mean(
    example_level_preactivation_original,
    original_meta,
    ALL_LABELS,
)
original_postactivation_label_mean, original_post_label_counts = compute_label_wise_mean(
    example_level_postactivation_original,
    original_meta,
    ALL_LABELS,
)
if original_post_label_counts != original_label_counts:
    raise RuntimeError("Original pre/post label counts disagree.")

original_label_mean = (
    original_preactivation_label_mean
    if activation_type == "pre"
    else original_postactivation_label_mean
)

target_condition_label_mean, target_label_counts = compute_label_wise_mean(
    example_level_score_target,
    target_meta,
    kept_labels,
)

original_label_firing_frequency, original_firing_counts = compute_label_wise_firing_frequency(
    example_level_fired_original,
    original_meta,
    ALL_LABELS,
)
target_condition_label_firing_frequency, target_firing_counts = compute_label_wise_firing_frequency(
    example_level_fired_target,
    target_meta,
    kept_labels,
)

target_label_to_local_row = {
    int(label): local_idx for local_idx, label in enumerate(kept_labels)
}

# Frequency-filter statistics are defined on the original target-label examples.
original_target_label_rows = get_rows_for_label(original_meta, target_label)
original_target_label_row_tensor = torch.tensor(
    original_target_label_rows,
    dtype=torch.long,
)
target_label_original_fire_count = (
    example_level_fired_original[original_target_label_row_tensor]
    .sum(dim=0)
    .to(torch.int64)
)
target_label_original_firing_frequency = (
    example_level_fired_original[original_target_label_row_tensor]
    .float()
    .mean(dim=0)
)
TARGET_LABEL_FREQUENCY_DENOMINATOR = len(original_target_label_rows)

print("original_preactivation_label_mean shape:", tuple(original_preactivation_label_mean.shape))
print("original_postactivation_label_mean shape:", tuple(original_postactivation_label_mean.shape))
print("original_label_mean shape:", tuple(original_label_mean.shape))
print("target_condition_label_mean shape:", tuple(target_condition_label_mean.shape))
print("original_label_firing_frequency shape:", tuple(original_label_firing_frequency.shape))
print("target_condition_label_firing_frequency shape:", tuple(target_condition_label_firing_frequency.shape))
print("Original counts:", original_label_counts)
print("Target counts:", target_label_counts)
print(
    "Frequency filter reference set:",
    f"{TARGET_LABEL_FREQUENCY_DENOMINATOR} original-condition "
    f"{LABEL_TO_WORD[target_label]} final Example-8 instances",
)


In [ ]:
# ============================================================
# 10. Rank removed-label-selective features and create volcano frequency filter
# ============================================================


def compute_target_selective_feature_table(
    original_means: torch.Tensor,
    removed_label: int,
) -> pd.DataFrame:
    target_values = original_means[int(removed_label)].detach().float().cpu()
    other_labels = [label for label in ALL_LABELS if label != int(removed_label)]
    other_values = original_means[other_labels].detach().float().cpu()

    max_other_values, max_other_local = other_values.max(dim=0)
    max_other_labels = torch.tensor(other_labels, dtype=torch.long)[max_other_local]
    global_max_values, preferred_labels = original_means.detach().float().cpu().max(dim=0)

    margin = target_values - max_other_values
    feature_indices = torch.arange(original_means.shape[1], dtype=torch.long)
    target_is_preferred = preferred_labels == int(removed_label)

    fire_count = target_label_original_fire_count.detach().cpu()
    fire_frequency = target_label_original_firing_frequency.detach().float().cpu()
    passes_frequency_filter = fire_frequency >= float(frequency_filter)

    table = pd.DataFrame({
        "feature_idx": feature_indices.numpy().astype(int),
        "activation_type": activation_type,
        "preferred_label": preferred_labels.numpy().astype(int),
        "preferred_label_name": [LABEL_TO_WORD[int(label)] for label in preferred_labels],
        "target_is_preferred": target_is_preferred.numpy().astype(bool),
        "target_original_mean": target_values.numpy(),
        "strongest_other_label": max_other_labels.numpy().astype(int),
        "strongest_other_label_name": [LABEL_TO_WORD[int(label)] for label in max_other_labels],
        "strongest_other_original_mean": max_other_values.numpy(),
        "selectivity_margin": margin.numpy(),
        "global_original_max": global_max_values.numpy(),
        "target_label_fire_count": fire_count.numpy().astype(int),
        "target_label_fire_frequency": fire_frequency.numpy(),
        "passes_frequency_filter": passes_frequency_filter.numpy().astype(bool),
    })

    return (
        table[table["target_is_preferred"]]
        .sort_values(["selectivity_margin", "target_original_mean"], ascending=False)
        .reset_index(drop=True)
    )


target_selective_feature_df = compute_target_selective_feature_table(
    original_label_mean,
    target_label,
)

# Preserve the original top-K selectivity definition.
top_target_selective_feature_df = target_selective_feature_df.head(
    TOPK_TARGET_SELECTIVE_FEATURES
).copy()
top_target_selective_features = (
    top_target_selective_feature_df["feature_idx"].astype(int).tolist()
)

if len(top_target_selective_features) == 0:
    raise RuntimeError(
        f"No features preferred the removed label {target_label} "
        f"({LABEL_TO_WORD[target_label]}) in the original condition."
    )

# The frequency filter is applied specifically to the rank-biserial/Wilcoxon plots.
volcano_feature_df = top_target_selective_feature_df[
    top_target_selective_feature_df["passes_frequency_filter"]
].copy().reset_index(drop=True)
volcano_feature_indices = volcano_feature_df["feature_idx"].astype(int).tolist()

if len(volcano_feature_indices) == 0:
    raise RuntimeError(
        f"None of the top {len(top_target_selective_features)} removed-label-selective "
        f"features passed frequency_filter={frequency_filter:.4f}. Lower the filter or "
        "increase TOPK_TARGET_SELECTIVE_FEATURES."
    )

print(
    f"Found {len(target_selective_feature_df)} features whose largest original "
    f"{ACTIVATION_VALUE_NAME} mean belongs to {LABEL_TO_WORD[target_label]}."
)
print(
    f"Using the top {len(top_target_selective_features)} by margin to the strongest other label."
)
print(
    f"Volcano frequency filter: fired in at least {frequency_filter:.2%} of "
    f"{TARGET_LABEL_FREQUENCY_DENOMINATOR} original {LABEL_TO_WORD[target_label]} instances."
)
print(
    f"Features retained for volcano plots: {len(volcano_feature_indices)} / "
    f"{len(top_target_selective_features)}."
)

display(top_target_selective_feature_df.head(20))
print("Frequency-eligible volcano features:")
display(volcano_feature_df.head(20))


In [ ]:
# ============================================================
# 11. Optional decoder-side antipodality for only the selected features
# ============================================================


def get_subset_feature_negations(
    W_dec: torch.Tensor,
    feature_indices: Sequence[int],
) -> pd.DataFrame:
    feature_indices = [int(feature_idx) for feature_idx in feature_indices]
    W = W_dec.detach().float().cpu()
    W = W / W.norm(dim=1, keepdim=True).clamp_min(1e-8)

    query = W[feature_indices]
    similarities = query @ W.T

    rows = []
    for local_idx, feature_idx in enumerate(feature_indices):
        similarities[local_idx, feature_idx] = float("inf")
        min_value, min_index = torch.min(similarities[local_idx], dim=0)
        rows.append({
            "feature_idx": int(feature_idx),
            "antipodal_partner": int(min_index.item()),
            "decoder_cosine_to_partner": float(min_value.item()),
        })
    return pd.DataFrame(rows)


if RUN_ANTIPODALITY_FOR_TOP_FEATURES:
    antipodality_df = get_subset_feature_negations(
        sae.W_dec,
        top_target_selective_features,
    )
else:
    antipodality_df = pd.DataFrame({
        "feature_idx": top_target_selective_features,
        "antipodal_partner": np.nan,
        "decoder_cosine_to_partner": np.nan,
    })

display(antipodality_df.head(20))


In [ ]:
# ============================================================
# 12. Feature-level AFR summary table
# ============================================================


def build_feature_afr_summary(feature_indices: Sequence[int]) -> pd.DataFrame:
    selectivity_lookup = target_selective_feature_df.set_index("feature_idx")
    antipodality_lookup = antipodality_df.set_index("feature_idx")
    rows = []

    for feature_idx in map(int, feature_indices):
        selectivity_row = selectivity_lookup.loc[feature_idx]
        row = {
            "feature_idx": feature_idx,
            "activation_type": activation_type,
            "selectivity_margin": float(selectivity_row["selectivity_margin"]),
            "target_original_mean": float(original_label_mean[target_label, feature_idx]),
            "target_label_fire_count": int(selectivity_row["target_label_fire_count"]),
            "target_label_fire_frequency": float(selectivity_row["target_label_fire_frequency"]),
            "passes_frequency_filter": bool(selectivity_row["passes_frequency_filter"]),
            "pooled_paired_shift_mean": float(paired_feature_shift[:, feature_idx].mean()),
            "pooled_paired_shift_median": float(paired_feature_shift[:, feature_idx].median()),
            "pooled_fraction_shift_positive": float(
                (paired_feature_shift[:, feature_idx] > 0).float().mean()
            ),
        }

        for label in ALL_LABELS:
            label_name = LABEL_TO_WORD[label].lower()
            row[f"original_{label_name}_mean"] = float(original_label_mean[label, feature_idx])
            row[f"original_{label_name}_firing_frequency"] = float(
                original_label_firing_frequency[label, feature_idx]
            )

            if label in kept_labels:
                local_idx = target_label_to_local_row[label]
                target_mean_value = float(
                    target_condition_label_mean[local_idx, feature_idx]
                )
                row[f"target_{label_name}_mean"] = target_mean_value
                row[f"target_{label_name}_firing_frequency"] = float(
                    target_condition_label_firing_frequency[local_idx, feature_idx]
                )
                row[f"delta_{label_name}"] = (
                    target_mean_value - float(original_label_mean[label, feature_idx])
                )
            else:
                row[f"target_{label_name}_mean"] = np.nan
                row[f"target_{label_name}_firing_frequency"] = np.nan
                row[f"delta_{label_name}"] = np.nan

        if feature_idx in antipodality_lookup.index:
            row["antipodal_partner"] = antipodality_lookup.loc[
                feature_idx, "antipodal_partner"
            ]
            row["decoder_cosine_to_partner"] = antipodality_lookup.loc[
                feature_idx, "decoder_cosine_to_partner"
            ]
        rows.append(row)

    return pd.DataFrame(rows).sort_values(
        ["selectivity_margin", "pooled_paired_shift_mean"],
        ascending=False,
    ).reset_index(drop=True)


feature_afr_summary_df = build_feature_afr_summary(top_target_selective_features)
display(feature_afr_summary_df.head(20))


## 7. Paired Wilcoxon / rank-biserial volcano analysis

The pooled comparison concatenates the one-to-one matched final examples from all three retained labels. Positive rank-biserial correlation means the feature tends to have a larger selected activation value in the target condition than for the same Example 8 in the original four-label condition.

The volcano candidate set starts from the top `TOPK_TARGET_SELECTIVE_FEATURES` removed-label-selective features and then applies `frequency_filter` using original-condition `target_label` firing frequency. The filter is based on actual feature-firing events under the configured SAE gate regardless of whether `activation_type` is `"pre"` or `"post"`.


In [ ]:
# ============================================================
# 13. Paired Wilcoxon / rank-biserial statistics
# ============================================================


def benjamini_hochberg(pvalues: Sequence[float]) -> np.ndarray:
    pvalues = np.asarray(pvalues, dtype=np.float64)
    qvalues = np.full_like(pvalues, np.nan)
    finite_mask = np.isfinite(pvalues)
    finite_indices = np.where(finite_mask)[0]
    if len(finite_indices) == 0:
        return qvalues

    finite_p = pvalues[finite_indices]
    order = np.argsort(finite_p)
    ranked = finite_p[order]
    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0.0, 1.0)

    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    qvalues[finite_indices] = restored
    return qvalues


def paired_rank_biserial_and_wilcoxon(diff_values: Sequence[float]) -> Dict[str, float]:
    diff = np.asarray(diff_values, dtype=np.float64)
    diff = diff[np.isfinite(diff)]
    nonzero = diff[diff != 0]

    if len(nonzero) == 0:
        return {
            "rank_biserial": 0.0,
            "pvalue": 1.0,
            "n_total": int(len(diff)),
            "n_nonzero": 0,
        }

    ranks = rankdata(np.abs(nonzero), method="average")
    w_plus = float(ranks[nonzero > 0].sum())
    w_minus = float(ranks[nonzero < 0].sum())
    denominator = w_plus + w_minus
    rank_biserial = (w_plus - w_minus) / denominator if denominator > 0 else 0.0

    try:
        pvalue = float(
            wilcoxon(
                nonzero,
                zero_method="wilcox",
                alternative="two-sided",
                method="approx",
            ).pvalue
        )
    except Exception:
        pvalue = float("nan")

    return {
        "rank_biserial": float(rank_biserial),
        "pvalue": pvalue,
        "n_total": int(len(diff)),
        "n_nonzero": int(len(nonzero)),
    }


def compute_volcano_results(
    original_aligned: torch.Tensor,
    target_scores: torch.Tensor,
    feature_indices: Sequence[int],
    row_mask: Optional[Sequence[bool]] = None,
) -> pd.DataFrame:
    if original_aligned.shape != target_scores.shape:
        raise ValueError("Original-aligned and target score matrices must have identical shapes")

    if row_mask is None:
        row_mask_np = np.ones(target_scores.shape[0], dtype=bool)
    else:
        row_mask_np = np.asarray(row_mask, dtype=bool)
        if row_mask_np.shape != (target_scores.shape[0],):
            raise ValueError("row_mask has the wrong shape")

    selected_rows = torch.tensor(np.where(row_mask_np)[0], dtype=torch.long)
    diff_matrix = (
        target_scores[selected_rows].detach().float().cpu()
        - original_aligned[selected_rows].detach().float().cpu()
    )

    selectivity_lookup = target_selective_feature_df.set_index("feature_idx")
    rows = []

    for feature_idx in map(int, feature_indices):
        selectivity_row = selectivity_lookup.loc[feature_idx]
        fire_frequency = float(selectivity_row["target_label_fire_frequency"])
        if fire_frequency + 1e-12 < float(frequency_filter):
            # Defensive check: plot candidates should already have been filtered.
            continue

        diff = diff_matrix[:, feature_idx].numpy()
        stats = paired_rank_biserial_and_wilcoxon(diff)
        rows.append({
            "feature_idx": feature_idx,
            "activation_type": activation_type,
            "rank_biserial": stats["rank_biserial"],
            "pvalue": stats["pvalue"],
            "n_total": stats["n_total"],
            "n_nonzero": stats["n_nonzero"],
            "mean_target_minus_original": float(np.mean(diff)),
            "median_target_minus_original": float(np.median(diff)),
            "fraction_positive": float(np.mean(diff > 0)),
            "selectivity_margin": float(selectivity_row["selectivity_margin"]),
            "removed_label_original_mean": float(
                original_label_mean[target_label, feature_idx]
            ),
            "target_label_fire_count": int(selectivity_row["target_label_fire_count"]),
            "target_label_fire_frequency": fire_frequency,
            "frequency_filter": float(frequency_filter),
        })

    result = pd.DataFrame(rows)
    if len(result) == 0:
        raise RuntimeError(
            "No features remained for the volcano analysis after applying the "
            f"frequency_filter={frequency_filter:.4f}."
        )

    result["qvalue_bh"] = benjamini_hochberg(result["pvalue"].to_numpy())
    return result.sort_values(
        ["rank_biserial", "mean_target_minus_original"],
        ascending=False,
    ).reset_index(drop=True)


def plot_volcano(
    result_df: pd.DataFrame,
    title: str,
    p_threshold: float = VOLCANO_P_THRESHOLD,
):
    df = result_df.copy()
    df["plot_pvalue"] = np.clip(df["pvalue"].astype(float), 1e-300, 1.0)
    df["group"] = "Not significant"
    df.loc[
        (df["pvalue"] < p_threshold) & (df["rank_biserial"] > 0),
        "group",
    ] = "Significant increase"
    df.loc[
        (df["pvalue"] < p_threshold) & (df["rank_biserial"] < 0),
        "group",
    ] = "Significant decrease"

    fig = go.Figure()
    for group_name in ["Not significant", "Significant decrease", "Significant increase"]:
        subset = df[df["group"] == group_name]
        fig.add_trace(go.Scatter(
            x=subset["rank_biserial"],
            y=subset["plot_pvalue"],
            mode="markers",
            name=group_name,
            text=[
                f"feature={int(row.feature_idx)}<br>"
                f"activation_type={row.activation_type}<br>"
                f"rank_biserial={row.rank_biserial:+.4f}<br>"
                f"mean_delta={row.mean_target_minus_original:+.4f}<br>"
                f"p={row.pvalue:.3e}<br>"
                f"BH q={row.qvalue_bh:.3e}<br>"
                f"selectivity_margin={row.selectivity_margin:+.4f}<br>"
                f"target-label firing={row.target_label_fire_count}/"
                f"{TARGET_LABEL_FREQUENCY_DENOMINATOR} "
                f"({row.target_label_fire_frequency:.2%})"
                for row in subset.itertuples(index=False)
            ],
            hoverinfo="text",
            marker=dict(size=8, opacity=0.75),
        ))

    fig.add_vline(x=0.0, line_width=2, line_dash="dash")
    fig.add_hline(y=p_threshold, line_width=2, line_dash="dash")
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            font=dict(family="Times New Roman", size=28),
        ),
        xaxis=dict(
            title="Rank-biserial correlation (target − original)",
            title_font=dict(family="Times New Roman", size=24),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        yaxis=dict(
            title="Two-sided Wilcoxon p-value",
            title_font=dict(family="Times New Roman", size=24),
            tickfont=dict(family="Times New Roman", size=14),
            type="log",
            autorange="reversed",
        ),
        width=950,
        height=650,
        margin=dict(l=120, r=60, t=135, b=100),
        legend=dict(font=dict(family="Times New Roman", size=14)),
    )
    return fig


pooled_volcano_results_df = compute_volcano_results(
    example_level_score_original_aligned,
    example_level_score_target,
    volcano_feature_indices,
)

pooled_volcano_fig = plot_volcano(
    pooled_volcano_results_df,
    title=(
        f"Adaptive Feature Reorganization in {MODEL_SHORT_NAME}<br>"
        f"<sup>AGNews | layer {TARGET_LAYER} | "
        f"removed label: {LABEL_TO_WORD[target_label]}</sup>"
    ),
)

print(
    f"Features retained for volcano plots: "
    f"{len(volcano_feature_indices)}"
)

pooled_volcano_fig.show()

display(pooled_volcano_results_df.head(25))


In [16]:
# ============================================================
# 14. Optional per-retained-label volcano helper
# ============================================================


def run_volcano_for_kept_label(label_id: int):
    label_id = int(label_id)
    if label_id not in kept_labels:
        raise ValueError(
            f"label_id must be one of retained labels {kept_labels}; got {label_id}"
        )

    row_mask = np.asarray([
        int(meta_item["final_label"]) == label_id
        for meta_item in target_meta
    ])

    result_df = compute_volcano_results(
        example_level_score_original_aligned,
        example_level_score_target,
        volcano_feature_indices,
        row_mask=row_mask,
    )
    fig = plot_volcano(
        result_df,
        title=(
            f"AFR on matched {LABEL_TO_WORD[label_id]} Example-8 articles<br>"
            f"<sup>Removed label: {LABEL_TO_WORD[target_label]} | "
            f"activation: {activation_type} | target-label firing ≥ "
            f"{frequency_filter:.1%}</sup>"
        ),
    )
    fig.show()
    display(result_df.head(20))
    return result_df


# Example:
# sports_only_volcano_df = run_volcano_for_kept_label(1)


## 8. Inspect any individual feature

`show_feature_label_shift(feature_idx)` reports:

- the original four-label average value under the selected `activation_type`;
- the target-condition mean for each retained label;
- target-minus-original paired shifts;
- original and target feature firing frequencies;
- per-label paired rank-biserial and Wilcoxon statistics.

The removed label has no target-condition row by design, so its target mean, target firing frequency, and shift are shown as missing.


In [ ]:
# ============================================================
# 15. Requested feature-index diagnostic
# ============================================================


def show_feature_label_shift(feature_idx: int, show_plots: bool = True) -> pd.DataFrame:
    """
    Display how one SAE feature behaves by final Example-8 label.

    The reported feature values follow the global `activation_type` setting.
    Firing frequencies always use actual SAE threshold crossings.

    Parameters
    ----------
    feature_idx:
        Integer SAE feature index.
    show_plots:
        If True, show a grouped mean-value plot and a paired-shift plot.

    Returns
    -------
    pd.DataFrame
        One row per AGNews label. The target-condition columns are NaN for the
        removed label because that label does not occur in the target prompt set.
    """
    feature_idx = int(feature_idx)
    if not (0 <= feature_idx < sae.n_features):
        raise ValueError(f"feature_idx must be in [0, {sae.n_features}); got {feature_idx}")

    original_mean_col = f"original_mean_{ACTIVATION_COLUMN_SUFFIX}"
    original_median_col = f"original_median_{ACTIVATION_COLUMN_SUFFIX}"
    target_mean_col = f"target_mean_{ACTIVATION_COLUMN_SUFFIX}"
    target_median_col = f"target_median_{ACTIVATION_COLUMN_SUFFIX}"

    rows = []
    for label in ALL_LABELS:
        original_indices = get_rows_for_label(original_meta, label)
        original_values = (
            example_level_score_original[original_indices, feature_idx]
            .detach()
            .float()
            .cpu()
            .numpy()
        )
        original_fired = (
            example_level_fired_original[original_indices, feature_idx]
            .detach()
            .cpu()
            .numpy()
            .astype(bool)
        )

        row = {
            "label_id": int(label),
            "label_name": LABEL_TO_WORD[label],
            "activation_type": activation_type,
            "n_original": int(len(original_indices)),
            original_mean_col: float(np.mean(original_values)),
            original_median_col: float(np.median(original_values)),
            "original_firing_frequency": float(np.mean(original_fired)),
            "original_fire_count": int(np.sum(original_fired)),
        }

        if label in kept_labels:
            target_indices = get_rows_for_label(target_meta, label)
            target_values = (
                example_level_score_target[target_indices, feature_idx]
                .detach()
                .float()
                .cpu()
                .numpy()
            )
            target_fired = (
                example_level_fired_target[target_indices, feature_idx]
                .detach()
                .cpu()
                .numpy()
                .astype(bool)
            )
            paired_original_values = (
                example_level_score_original_aligned[target_indices, feature_idx]
                .detach()
                .float()
                .cpu()
                .numpy()
            )
            paired_delta = target_values - paired_original_values
            stats = paired_rank_biserial_and_wilcoxon(paired_delta)

            row.update({
                "n_target": int(len(target_indices)),
                target_mean_col: float(np.mean(target_values)),
                target_median_col: float(np.median(target_values)),
                "target_firing_frequency": float(np.mean(target_fired)),
                "target_fire_count": int(np.sum(target_fired)),
                "target_minus_original_mean": float(np.mean(paired_delta)),
                "target_minus_original_median": float(np.median(paired_delta)),
                "fraction_paired_increase": float(np.mean(paired_delta > 0)),
                "paired_rank_biserial": float(stats["rank_biserial"]),
                "paired_wilcoxon_p": float(stats["pvalue"]),
            })
        else:
            row.update({
                "n_target": 0,
                target_mean_col: np.nan,
                target_median_col: np.nan,
                "target_firing_frequency": np.nan,
                "target_fire_count": 0,
                "target_minus_original_mean": np.nan,
                "target_minus_original_median": np.nan,
                "fraction_paired_increase": np.nan,
                "paired_rank_biserial": np.nan,
                "paired_wilcoxon_p": np.nan,
            })

        rows.append(row)

    result_df = pd.DataFrame(rows)

    target_fire_frequency = float(
        target_label_original_firing_frequency[feature_idx]
    )
    target_fire_count = int(target_label_original_fire_count[feature_idx])
    passes_filter = target_fire_frequency + 1e-12 >= float(frequency_filter)

    print("=" * 100)
    print(f"Feature {feature_idx}")
    print(f"activation_type: {activation_type} ({ACTIVATION_VALUE_NAME})")
    print(f"Removed label: {target_label} = {LABEL_TO_WORD[target_label]}")
    print(
        f"Original {LABEL_TO_WORD[target_label]} firing frequency: "
        f"{target_fire_count}/{TARGET_LABEL_FREQUENCY_DENOMINATOR} "
        f"({target_fire_frequency:.2%})"
    )
    print(
        f"Passes volcano frequency_filter={frequency_filter:.2%}: {passes_filter}"
    )
    if feature_idx in set(target_selective_feature_df["feature_idx"].astype(int)):
        selectivity_row = target_selective_feature_df.set_index("feature_idx").loc[feature_idx]
        print(
            "Original removed-label selectivity margin: "
            f"{selectivity_row['selectivity_margin']:+.6f}"
        )
    print("=" * 100)
    display(result_df)

    if show_plots:
        fig_means = go.Figure()
        fig_means.add_trace(go.Bar(
            x=result_df["label_name"],
            y=result_df[original_mean_col],
            name="Original: four labels",
            text=[f"{value:+.3f}" for value in result_df[original_mean_col]],
            textposition="auto",
        ))
        fig_means.add_trace(go.Bar(
            x=result_df["label_name"],
            y=result_df[target_mean_col],
            name=f"Target: without {LABEL_TO_WORD[target_label]}",
            text=[
                "" if not np.isfinite(value) else f"{value:+.3f}"
                for value in result_df[target_mean_col]
            ],
            textposition="auto",
        ))
        fig_means.update_layout(
            barmode="group",
            title=dict(
                text=(
                    f"Feature {feature_idx}: average {ACTIVATION_VALUE_NAME} "
                    "by final Example-8 label"
                ),
                x=0.5,
                font=dict(family="Times New Roman", size=26),
            ),
            xaxis=dict(
                title="Final Example-8 label",
                title_font=dict(family="Times New Roman", size=22),
                tickfont=dict(family="Times New Roman", size=16),
            ),
            yaxis=dict(
                title=f"Average {ACTIVATION_AXIS_LABEL}",
                title_font=dict(family="Times New Roman", size=22),
                tickfont=dict(family="Times New Roman", size=14),
            ),
            width=950,
            height=600,
            legend=dict(font=dict(family="Times New Roman", size=14)),
        )
        fig_means.show()

        retained_df = result_df[result_df["label_id"].isin(kept_labels)].copy()
        fig_delta = go.Figure(go.Bar(
            x=retained_df["label_name"],
            y=retained_df["target_minus_original_mean"],
            text=[f"{value:+.3f}" for value in retained_df["target_minus_original_mean"]],
            textposition="auto",
            customdata=np.stack([
                retained_df["paired_rank_biserial"].to_numpy(),
                retained_df["paired_wilcoxon_p"].to_numpy(),
                retained_df["target_firing_frequency"].to_numpy(),
            ], axis=1),
            hovertemplate=(
                "label=%{x}<br>mean paired shift=%{y:+.4f}<br>"
                "rank-biserial=%{customdata[0]:+.4f}<br>"
                "Wilcoxon p=%{customdata[1]:.3e}<br>"
                "target firing frequency=%{customdata[2]:.2%}<extra></extra>"
            ),
        ))
        fig_delta.add_hline(y=0.0, line_dash="dash")
        fig_delta.update_layout(
            title=dict(
                text=(
                    f"Feature {feature_idx}: target − original paired "
                    f"{ACTIVATION_VALUE_NAME} shift after removing "
                    f"{LABEL_TO_WORD[target_label]}"
                ),
                x=0.5,
                font=dict(family="Times New Roman", size=24),
            ),
            xaxis=dict(
                title="Retained final Example-8 label",
                title_font=dict(family="Times New Roman", size=22),
                tickfont=dict(family="Times New Roman", size=16),
            ),
            yaxis=dict(
                title=f"Mean paired {ACTIVATION_VALUE_NAME} shift",
                title_font=dict(family="Times New Roman", size=22),
                tickfont=dict(family="Times New Roman", size=14),
            ),
            width=900,
            height=560,
        )
        fig_delta.show()

    return result_df


# Inspect the strongest removed-label-selective feature that also passes the
# volcano frequency filter by default.
FEATURE_INDEX_TO_INSPECT = int(volcano_feature_indices[0])
feature_label_shift_df = show_feature_label_shift(FEATURE_INDEX_TO_INSPECT)

# For any other feature, run for example:
# feature_label_shift_df = show_feature_label_shift(1234)


## Interpretation of the activation-level experiment

The primary activation-level AFR signal is a **positive paired target-minus-original shift** for features that were originally selective for the removed label. Because Example 8 is exactly matched, the paired contrast asks how the same final article/label representation changes when the prompt's label inventory and two removed-label context examples are altered.

`activation_type` determines whether this shift is measured before or after the configured sparse gate. `frequency_filter` does not use the selected activation value; it always uses actual threshold-crossing frequency on original-condition `target_label` instances, ensuring that the volcano plot excludes features that almost never fired for the label they nominally preferred.

The following section extends this representational result with the paper's **causal feature-steering experiment**.


# 9. Causal AFR with independent causal screening and held-out evaluation

This section tests whether features exhibiting strong activation-level AFR also reorganize their causal effects, while separating **causal feature screening** from the **final held-out evaluation**.

For each primary/removed label \(L\):

1. **Activation-level feature selection.** As above, a feature must have its largest original mean preactivation on \(L\), pass the original-\(L\) firing-frequency threshold, and rank among the top 10 positive activation-AFR features.
2. **Independent causal-screening prompts.** All 40 activation-selected features are tested one at a time on a new set of AGNews test queries that is disjoint from the activation-discovery queries. A feature passes the causal-specificity screen only if
   \[
   \mathrm{CE}^{\mathrm{screen}}_{f\to L}>0
   \qquad\text{and}\qquad
   \mathrm{CE}^{\mathrm{screen}}_{f\to L}
   >
   \max_{c\neq L}\mathrm{CE}^{\mathrm{screen}}_{f\to c}.
   \]
   The screen uses only original four-label prompts; no label-exclusion outcome is used for selection.
3. **Held-out causal evaluation.** Only features passing the screen are evaluated on a second, query-disjoint prompt set. The final red/blue/green bars are computed exclusively from these held-out prompts:
   \[
   \underbrace{\max_{c\neq L}\mathrm{CE}^{\mathrm{holdout,orig}}_{f\to c}}_{\text{red}},
   \qquad
   \underbrace{\max_{L'\neq L}\mathrm{CE}^{\mathrm{holdout,ctrl}(-L)}_{f\to L'}}_{\text{blue}},
   \qquad
   \underbrace{\mathrm{CE}^{\mathrm{holdout,orig}}_{f\to L}}_{\text{green}}.
   \]

By default, the activation-discovery, causal-screening, and held-out query pools each contain 100 examples per label and are mutually disjoint. The causal prompt still ends exactly at the user-side `Topic:\n` boundary, and all output probabilities are normalized over the same four AGNews label tokens.

In [ ]:
# ============================================================
# 16. User-editable causal AFR configuration
# ============================================================

import inspect

RUN_AFR_CAUSAL_EXPERIMENT = True

# Reproduce all four primary/removed-label groups in the paper figure.
CAUSAL_REMOVED_LABELS = list(ALL_LABELS)

# Top activation-AFR features tested per primary label: 4 * 10 = 40 features.
CAUSAL_TOP_FEATURES_PER_LABEL = 15

# -------------------------------------------------------------------------
# Three query-disjoint stages
# -------------------------------------------------------------------------
# Activation-AFR discovery continues to use the already constructed labeled
# activation prompts. Causal screening and final evaluation use NEW AGNews test
# queries that exclude every final query used by the activation analysis.
CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL = NUM_FINAL_EXAMPLES_PER_LABEL
CAUSAL_SCREENING_EXAMPLES_PER_LABEL = NUM_FINAL_EXAMPLES_PER_LABEL
CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL = NUM_FINAL_EXAMPLES_PER_LABEL

# Independent deterministic seeds for causal final-query sampling and contexts.
CAUSAL_QUERY_POOL_SEED = SEED + 701_003
CAUSAL_CONTEXT_SEED_BASE = SEED + 702_007

# Eligibility criterion before activation-AFR ranking.
CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY = frequency_filter
CAUSAL_AFR_RANK_ACTIVATION_TYPE = activation_type  # "pre" or "post"
CAUSAL_REQUIRE_POSITIVE_AFR = True
CAUSAL_AFR_DISCOVERY_BATCH_SIZE = BATCH_SIZE

# Causal-screening criteria. Both are enabled for the intended experiment:
#   green_screen > 0
#   green_screen > red_screen
CAUSAL_SCREEN_REQUIRE_PRIMARY_POSITIVE = True
CAUSAL_SCREEN_REQUIRE_PRIMARY_GT_NONPRIMARY = True

# Constant steering coefficient used for EVERY feature and selected token:
#     hidden <- hidden + CAUSAL_ALPHA * W_dec[feature_idx]
CAUSAL_ALPHA = 50.0

# Add the raw decoder/write direction. Set True only for a unit-direction control.
CAUSAL_NORMALIZE_DECODER_DIRECTION = False

# "article":    final Example-8 article text only.
# "full_query": Article:, article text, and Topic: (not the Example 8 header).
# "last_k":     end-relative positions in the complete unlabeled causal prefix.
CAUSAL_TOKEN_SCOPE = "full_query"
CAUSAL_LAST_K = [-1, -2]

# This notebook's causal output-scoring prefix ends exactly at user-side Topic:\n.
CAUSAL_PROMPT_ENDING_MODE = "topic_only"
CAUSAL_LABEL_COMPLETION_PREFIX = ""

# Runtime / live-dashboard controls.
CAUSAL_BATCH_SIZE = 4
CAUSAL_SHOW_SCREENING_RESULTS = True
CAUSAL_SHOW_HOLDOUT_RESULTS = True
CAUSAL_CLEAR_OUTPUT_BETWEEN_FEATURES = True

if not np.isfinite(float(CAUSAL_ALPHA)):
    raise ValueError("CAUSAL_ALPHA must be a finite real number.")
if CAUSAL_TOKEN_SCOPE not in {"article", "full_query", "last_k"}:
    raise ValueError(
        "CAUSAL_TOKEN_SCOPE must be 'article', 'full_query', or 'last_k'."
    )
if CAUSAL_PROMPT_ENDING_MODE not in {"topic_only", "assistant_topic"}:
    raise ValueError(
        "CAUSAL_PROMPT_ENDING_MODE must be 'topic_only' or 'assistant_topic'."
    )

CAUSAL_AFR_RANK_ACTIVATION_TYPE = str(
    CAUSAL_AFR_RANK_ACTIVATION_TYPE
).strip().lower()
if CAUSAL_AFR_RANK_ACTIVATION_TYPE not in {"pre", "post"}:
    raise ValueError(
        "CAUSAL_AFR_RANK_ACTIVATION_TYPE must be 'pre' or 'post'."
    )

if not isinstance(CAUSAL_LAST_K, (list, tuple)) or len(CAUSAL_LAST_K) == 0:
    raise ValueError(
        "CAUSAL_LAST_K must be a non-empty list/tuple of negative integer "
        "offsets, e.g. [-1, -2]."
    )

_normalized_causal_last_k = []
for _causal_offset in CAUSAL_LAST_K:
    if isinstance(_causal_offset, bool) or not isinstance(
        _causal_offset, (int, np.integer)
    ):
        raise TypeError(
            "Every CAUSAL_LAST_K entry must be a negative integer; "
            f"got {_causal_offset!r}."
        )
    _causal_offset = int(_causal_offset)
    if _causal_offset >= 0:
        raise ValueError(
            "CAUSAL_LAST_K offsets are counted backward from the label-prediction "
            f"boundary, so every entry must be negative; got {_causal_offset}."
        )
    _normalized_causal_last_k.append(_causal_offset)

if len(set(_normalized_causal_last_k)) != len(_normalized_causal_last_k):
    raise ValueError(
        f"CAUSAL_LAST_K contains duplicate offsets: {_normalized_causal_last_k}."
    )
CAUSAL_LAST_K = _normalized_causal_last_k

del _normalized_causal_last_k, _causal_offset

if not all(int(label) in ALL_LABELS for label in CAUSAL_REMOVED_LABELS):
    raise ValueError(f"Invalid CAUSAL_REMOVED_LABELS={CAUSAL_REMOVED_LABELS}")
if not (0.0 <= float(CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY) <= 1.0):
    raise ValueError("CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY must lie in [0, 1].")
if int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL) < 1:
    raise ValueError("CAUSAL_SCREENING_EXAMPLES_PER_LABEL must be positive.")
if int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL) < 1:
    raise ValueError("CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL must be positive.")
if not (
    1 <= int(CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL)
    <= int(NUM_FINAL_EXAMPLES_PER_LABEL)
):
    raise ValueError(
        "CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL must lie between 1 and "
        f"NUM_FINAL_EXAMPLES_PER_LABEL={NUM_FINAL_EXAMPLES_PER_LABEL}."
    )

print("CAUSAL_PROMPT_ENDING_MODE:", CAUSAL_PROMPT_ENDING_MODE)
print("CAUSAL_TOKEN_SCOPE:", CAUSAL_TOKEN_SCOPE)
print("CAUSAL_LAST_K:", CAUSAL_LAST_K)
print(
    "Causal query counts per label: screening=",
    CAUSAL_SCREENING_EXAMPLES_PER_LABEL,
    ", holdout=",
    CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL,
)
print(
    "Causal screen:",
    "primary > 0 =", CAUSAL_SCREEN_REQUIRE_PRIMARY_POSITIVE,
    ", primary > strongest non-primary =",
    CAUSAL_SCREEN_REQUIRE_PRIMARY_GT_NONPRIMARY,
)

In [ ]:
# ============================================================
# 17. Build query-disjoint causal-screening and held-out prompt sets
# ============================================================

CAUSAL_QUERY_START_SENTINEL = "ZXQ_CAUSAL_QUERY_START_83d2a1f7"
CAUSAL_QUERY_END_SENTINEL = "ZXQ_CAUSAL_QUERY_END_83d2a1f7"
CAUSAL_GENERATION_SUFFIX = None


def example_from_key(key: str) -> Dict[str, Any]:
    split, index_text = str(key).split(":", 1)
    index = int(index_text)
    if split == "train":
        return dataset_example(train_data_agnews, index, "train")
    if split == "test":
        return dataset_example(test_data_agnews, index, "test")
    raise ValueError(f"Unknown example split in key={key!r}")


def reconstruct_examples_from_meta(meta_item: Dict[str, Any]) -> List[Dict[str, Any]]:
    examples = [example_from_key(key) for key in meta_item["example_keys"]]
    if len(examples) != N_EXAMPLES_PER_PROMPT:
        raise RuntimeError(
            f"Expected {N_EXAMPLES_PER_PROMPT} reconstructed examples; got {len(examples)}"
        )
    return examples


def build_controlled_examples_for_removed_label(
    original_examples_8: Sequence[Dict[str, Any]],
    pair_id: int,
    removed_label: int,
):
    """Replace exactly the two removed-label context examples; keep Example 8 fixed."""
    removed_label = int(removed_label)
    local_kept_labels = [label for label in ALL_LABELS if label != removed_label]

    final_example = dict(original_examples_8[-1])
    final_label = int(final_example["label"])
    if final_label == removed_label:
        raise ValueError(
            "A controlled label-exclusion prompt cannot use a final query from "
            "the removed label."
        )

    original_context = [dict(example) for example in original_examples_8[:-1]]
    removed_positions = [
        pos
        for pos, example in enumerate(original_context)
        if int(example["label"]) == removed_label
    ]
    if len(removed_positions) != 2:
        raise AssertionError(
            f"Expected exactly two removed-label context examples; got {removed_positions}"
        )

    rng = random.Random(SEED + 30_011 * (int(pair_id) + 1) + 101 * removed_label)
    replacement_labels = list(local_kept_labels)
    rng.shuffle(replacement_labels)
    replacement_labels = replacement_labels[:2]

    used_train_indices = {
        int(example["dataset_index"])
        for example in original_context
        if example["split"] == "train"
    }

    controlled_context = [dict(example) for example in original_context]
    replacement_records = []

    for position, replacement_label in zip(removed_positions, replacement_labels):
        replacement_idx = sample_indices_excluding(
            train_by_label[replacement_label],
            1,
            rng,
            exclude=used_train_indices,
        )[0]
        used_train_indices.add(replacement_idx)
        replacement_example = dataset_example(
            train_data_agnews,
            replacement_idx,
            "train",
        )
        controlled_context[position] = replacement_example
        replacement_records.append({
            "position_1indexed": int(position + 1),
            "replacement_label": int(replacement_label),
            "replacement_label_name": LABEL_TO_WORD[int(replacement_label)],
            "replacement_train_index": int(replacement_idx),
        })

    controlled_examples_8 = controlled_context + [final_example]
    counts = label_counts(controlled_examples_8)
    if counts[removed_label] != 0:
        raise AssertionError(f"Removed label remains in controlled prompt: {counts}")
    if sorted(counts[label] for label in local_kept_labels) != [2, 3, 3]:
        raise AssertionError(f"Controlled retained-label counts are invalid: {counts}")

    shared_context_positions = [
        position
        for position in range(7)
        if example_key(original_context[position])
        == example_key(controlled_context[position])
    ]
    if len(shared_context_positions) != 5:
        raise AssertionError(
            f"Expected five shared context positions; got {shared_context_positions}"
        )

    return controlled_examples_8, replacement_records, shared_context_positions


def causal_instruction_for_allowed_labels(label_ids: Sequence[int]) -> str:
    first_sentence = (
        f"The possible news topic labels are {format_label_list(label_ids)}."
    )
    second_sentence = (
        "The first seven examples contain an article followed by its correct "
        "topic label. Classify the eighth article by providing exactly one of "
        "the possible labels."
    )
    return first_sentence + " " + second_sentence


def render_unlabeled_causal_prompt(
    examples_8: Sequence[Dict[str, Any]],
    allowed_labels: Sequence[int],
) -> Tuple[str, Tuple[int, int], Tuple[int, int], str]:
    """Render seven labeled demonstrations plus one unlabeled Example-8 query."""
    if len(examples_8) != N_EXAMPLES_PER_PROMPT:
        raise ValueError(f"Expected 8 examples, got {len(examples_8)}")

    instruction = causal_instruction_for_allowed_labels(allowed_labels)
    first_sentence = instruction.split(". ", 1)[0] + "."
    demo_blocks = [
        format_labeled_example(example, idx + 1)
        for idx, example in enumerate(examples_8[:-1])
    ]

    final_example = examples_8[-1]
    final_text = clean_agnews_text(final_example["text"])
    query_content = f"Article:\n{final_text}\nTopic:\n"
    marked_query_block = (
        "Example 8\n"
        + CAUSAL_QUERY_START_SENTINEL
        + query_content
        + CAUSAL_QUERY_END_SENTINEL
    )
    user_content_marked = (
        instruction
        + "\n\n"
        + "\n\n".join(demo_blocks + [marked_query_block])
    )

    rendered_marked = apply_model_chat_template(
        [{"role": "user", "content": user_content_marked}],
        add_generation_prompt=True,
    )

    start_pos = rendered_marked.find(CAUSAL_QUERY_START_SENTINEL)
    end_pos = rendered_marked.find(CAUSAL_QUERY_END_SENTINEL)
    if start_pos < 0 or end_pos < 0 or end_pos <= start_pos:
        raise ValueError("Could not recover causal query sentinels after rendering.")

    rendered_query = rendered_marked[
        start_pos + len(CAUSAL_QUERY_START_SENTINEL):end_pos
    ]
    if rendered_query != query_content:
        raise ValueError("Chat template changed the marked causal query content.")

    template_prefix = rendered_marked[:start_pos]
    template_suffix = rendered_marked[
        end_pos + len(CAUSAL_QUERY_END_SENTINEL):
    ]

    global CAUSAL_GENERATION_SUFFIX
    if CAUSAL_GENERATION_SUFFIX is None:
        CAUSAL_GENERATION_SUFFIX = template_suffix
    elif template_suffix != CAUSAL_GENERATION_SUFFIX:
        raise RuntimeError(
            "The chat-template generation suffix changed across prompts: "
            f"{CAUSAL_GENERATION_SUFFIX!r} vs. {template_suffix!r}."
        )

    if CAUSAL_PROMPT_ENDING_MODE == "topic_only":
        prompt = template_prefix + query_content
    elif CAUSAL_PROMPT_ENDING_MODE == "assistant_topic":
        prompt = template_prefix + query_content + template_suffix + "Topic:\n"
    else:
        raise ValueError(CAUSAL_PROMPT_ENDING_MODE)

    query_char_span = (start_pos, start_pos + len(query_content))
    article_char_start = start_pos + len("Article:\n")
    article_char_span = (
        article_char_start,
        article_char_start + len(final_text),
    )

    if prompt[query_char_span[0]:query_char_span[1]] != query_content:
        raise ValueError("Causal query character-span mapping failed.")
    if prompt[article_char_span[0]:article_char_span[1]] != final_text:
        raise ValueError("Causal article character-span mapping failed.")

    exact_expected_ending = (
        "Topic:\n"
        if CAUSAL_PROMPT_ENDING_MODE == "topic_only"
        else f"Topic:\n{template_suffix}Topic:\n"
    )
    if not prompt.endswith(exact_expected_ending):
        raise RuntimeError(
            "Causal prompt does not have the requested exact ending. Expected "
            f"{exact_expected_ending!r}."
        )

    return prompt, query_char_span, article_char_span, first_sentence


def sample_query_disjoint_causal_final_examples():
    """
    Sample two new query pools from AGNews test examples unused by activation AFR.

    Returns a nested dictionary:
        result["screening"][label] -> list of examples
        result["holdout"][label]   -> list of examples
    """
    activation_used = {
        int(label): {
            int(example["dataset_index"])
            for example in final_examples_by_label[int(label)]
        }
        for label in ALL_LABELS
    }

    total_needed = (
        int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL)
        + int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
    )
    output = {
        "screening": {label: [] for label in ALL_LABELS},
        "holdout": {label: [] for label in ALL_LABELS},
    }

    for label in ALL_LABELS:
        available = [
            int(index)
            for index in test_by_label[int(label)]
            if int(index) not in activation_used[int(label)]
        ]
        if len(available) < total_needed:
            raise ValueError(
                f"Label {LABEL_TO_WORD[int(label)]} has only {len(available)} "
                f"unused test examples; need {total_needed}."
            )

        rng = random.Random(CAUSAL_QUERY_POOL_SEED + 10_003 * (int(label) + 1))
        chosen = rng.sample(available, total_needed)
        screen_indices = chosen[: int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL)]
        holdout_indices = chosen[int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL):]

        output["screening"][int(label)] = [
            dataset_example(test_data_agnews, index, "test")
            for index in screen_indices
        ]
        output["holdout"][int(label)] = [
            dataset_example(test_data_agnews, index, "test")
            for index in holdout_indices
        ]

    return output


def build_causal_original_examples(
    final_example: Dict[str, Any],
    pair_id: int,
    causal_split: str,
) -> List[Dict[str, Any]]:
    """Construct a balanced 2/2/2/2 eight-example prompt for a new causal query."""
    final_label = int(final_example["label"])
    desired_context_counts = {
        label: ORIGINAL_EXAMPLES_PER_LABEL for label in ALL_LABELS
    }
    desired_context_counts[final_label] -= 1

    split_salt = 0 if causal_split == "screening" else 1_000_003
    rng = random.Random(
        CAUSAL_CONTEXT_SEED_BASE
        + split_salt
        + 10_007 * (int(pair_id) + 1)
    )

    context = []
    used_train_indices = set()
    for label in ALL_LABELS:
        count = int(desired_context_counts[label])
        selected = sample_indices_excluding(
            train_by_label[label],
            count,
            rng,
            exclude=used_train_indices,
        )
        used_train_indices.update(selected)
        context.extend(
            dataset_example(train_data_agnews, index, "train")
            for index in selected
        )

    rng.shuffle(context)
    examples_8 = context + [dict(final_example)]
    expected_counts = {
        label: ORIGINAL_EXAMPLES_PER_LABEL for label in ALL_LABELS
    }
    if label_counts(examples_8) != expected_counts:
        raise AssertionError("Causal original prompt is not balanced 2/2/2/2.")
    return examples_8


causal_final_examples_by_split = sample_query_disjoint_causal_final_examples()


def build_causal_base_records() -> List[Dict[str, Any]]:
    records = []
    pair_id = 1_000_000
    for causal_split in ["screening", "holdout"]:
        for final_label in ALL_LABELS:
            examples = causal_final_examples_by_split[causal_split][int(final_label)]
            for within_label_idx, final_example in enumerate(examples):
                original_examples_8 = build_causal_original_examples(
                    final_example,
                    pair_id=pair_id,
                    causal_split=causal_split,
                )
                records.append({
                    "causal_split": causal_split,
                    "pair_id": int(pair_id),
                    "within_label_idx": int(within_label_idx),
                    "final_label": int(final_label),
                    "final_label_name": LABEL_TO_WORD[int(final_label)],
                    "final_example_key": example_key(final_example),
                    "final_text_hash": final_text_hash(final_example),
                    "original_examples_8": original_examples_8,
                })
                pair_id += 1
    return records


causal_base_records = build_causal_base_records()


def build_causal_prompt_table() -> pd.DataFrame:
    rows = []
    causal_prompt_id = 0

    for removed_label in CAUSAL_REMOVED_LABELS:
        removed_label = int(removed_label)
        local_kept_labels = [label for label in ALL_LABELS if label != removed_label]

        for base in causal_base_records:
            causal_split = str(base["causal_split"])
            final_label = int(base["final_label"])
            original_examples_8 = base["original_examples_8"]

            (
                original_prompt,
                original_query_span,
                original_article_span,
                original_first_sentence,
            ) = render_unlabeled_causal_prompt(
                original_examples_8,
                allowed_labels=ALL_LABELS,
            )

            rows.append({
                "causal_prompt_id": int(causal_prompt_id),
                "causal_split": causal_split,
                "removed_label": removed_label,
                "removed_label_name": LABEL_TO_WORD[removed_label],
                "condition": "original",
                "causal_prompt_ending_mode": CAUSAL_PROMPT_ENDING_MODE,
                "pair_id": int(base["pair_id"]),
                "within_label_idx": int(base["within_label_idx"]),
                "gold_label": final_label,
                "gold_label_name": LABEL_TO_WORD[final_label],
                "final_example_key": base["final_example_key"],
                "final_text_hash": base["final_text_hash"],
                "allowed_labels": tuple(ALL_LABELS),
                "instruction_first_sentence": original_first_sentence,
                "prompt": original_prompt,
                "query_char_span": tuple(map(int, original_query_span)),
                "article_char_span": tuple(map(int, original_article_span)),
                "prompt_hash": hashlib.sha1(
                    original_prompt.encode("utf-8")
                ).hexdigest(),
            })
            causal_prompt_id += 1

            # Screening deliberately uses only original four-label prompts.
            if causal_split != "holdout" or final_label == removed_label:
                continue

            controlled_examples_8, replacements, shared_positions = (
                build_controlled_examples_for_removed_label(
                    original_examples_8,
                    pair_id=int(base["pair_id"]),
                    removed_label=removed_label,
                )
            )
            (
                controlled_prompt,
                controlled_query_span,
                controlled_article_span,
                controlled_first_sentence,
            ) = render_unlabeled_causal_prompt(
                controlled_examples_8,
                allowed_labels=local_kept_labels,
            )

            rows.append({
                "causal_prompt_id": int(causal_prompt_id),
                "causal_split": causal_split,
                "removed_label": removed_label,
                "removed_label_name": LABEL_TO_WORD[removed_label],
                "condition": "controlled",
                "causal_prompt_ending_mode": CAUSAL_PROMPT_ENDING_MODE,
                "pair_id": int(base["pair_id"]),
                "within_label_idx": int(base["within_label_idx"]),
                "gold_label": final_label,
                "gold_label_name": LABEL_TO_WORD[final_label],
                "final_example_key": base["final_example_key"],
                "final_text_hash": base["final_text_hash"],
                "allowed_labels": tuple(local_kept_labels),
                "instruction_first_sentence": controlled_first_sentence,
                "replacement_records": replacements,
                "shared_context_positions_1indexed": tuple(
                    int(position + 1) for position in shared_positions
                ),
                "prompt": controlled_prompt,
                "query_char_span": tuple(map(int, controlled_query_span)),
                "article_char_span": tuple(map(int, controlled_article_span)),
                "prompt_hash": hashlib.sha1(
                    controlled_prompt.encode("utf-8")
                ).hexdigest(),
            })
            causal_prompt_id += 1

    table = pd.DataFrame(rows)

    for removed_label in CAUSAL_REMOVED_LABELS:
        removed_label = int(removed_label)

        screen = table[
            (table["causal_split"] == "screening")
            & (table["removed_label"] == removed_label)
        ]
        if len(screen) != 4 * int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL):
            raise AssertionError("Unexpected causal-screening prompt count.")
        if set(screen["condition"]) != {"original"}:
            raise AssertionError("Screening must contain original prompts only.")

        holdout = table[
            (table["causal_split"] == "holdout")
            & (table["removed_label"] == removed_label)
        ]
        original_sub = holdout[holdout["condition"] == "original"]
        controlled_sub = holdout[holdout["condition"] == "controlled"]
        expected_original = 4 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
        expected_controlled = 3 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
        if len(original_sub) != expected_original:
            raise AssertionError("Unexpected held-out original prompt count.")
        if len(controlled_sub) != expected_controlled:
            raise AssertionError("Unexpected held-out controlled prompt count.")
        if (controlled_sub["gold_label"] == removed_label).any():
            raise AssertionError("Controlled holdout contains the removed gold label.")

        original_pair_lookup = original_sub.set_index("pair_id")
        for controlled_row in controlled_sub.itertuples(index=False):
            original_row = original_pair_lookup.loc[int(controlled_row.pair_id)]
            if original_row["final_example_key"] != controlled_row.final_example_key:
                raise AssertionError("Held-out matched query identity failed.")
            if original_row["final_text_hash"] != controlled_row.final_text_hash:
                raise AssertionError("Held-out matched query text failed.")
            if int(original_row["gold_label"]) != int(controlled_row.gold_label):
                raise AssertionError("Held-out matched gold label failed.")
            if len(controlled_row.shared_context_positions_1indexed) != 5:
                raise AssertionError("Held-out context matching failed.")

    return table.reset_index(drop=True)


causal_prompt_df = build_causal_prompt_table()
causal_screening_prompt_df = causal_prompt_df[
    causal_prompt_df["causal_split"] == "screening"
].reset_index(drop=True)
causal_holdout_prompt_df = causal_prompt_df[
    causal_prompt_df["causal_split"] == "holdout"
].reset_index(drop=True)

# Strong disjointness audit at the final-query level.
activation_query_keys = {
    str(item["final_example_key"]) for item in original_meta
}
screen_query_keys = set(causal_screening_prompt_df["final_example_key"])
holdout_query_keys = set(causal_holdout_prompt_df["final_example_key"])
if activation_query_keys & screen_query_keys:
    raise AssertionError("Activation and causal-screening query pools overlap.")
if activation_query_keys & holdout_query_keys:
    raise AssertionError("Activation and held-out query pools overlap.")
if screen_query_keys & holdout_query_keys:
    raise AssertionError("Causal-screening and held-out query pools overlap.")

print("Causal prompt counts by split / removed label / condition / gold label")
display(
    causal_prompt_df.groupby([
        "causal_split",
        "removed_label_name",
        "condition",
        "gold_label_name",
    ]).size().rename("n").reset_index()
)
print("Screening prompt rows:", len(causal_screening_prompt_df))
print("Held-out prompt rows:", len(causal_holdout_prompt_df))
print("Unique screening prompt strings:", causal_screening_prompt_df["prompt_hash"].nunique())
print("Unique held-out prompt strings:", causal_holdout_prompt_df["prompt_hash"].nunique())
print("PASS: activation, screening, and holdout final-query pools are disjoint.")

if CAUSAL_PROMPT_ENDING_MODE == "topic_only":
    CAUSAL_EXACT_PROMPT_ENDING = "Topic:\n"
else:
    if CAUSAL_GENERATION_SUFFIX is None:
        raise RuntimeError("Causal generation suffix was not initialized.")
    CAUSAL_EXACT_PROMPT_ENDING = (
        f"Topic:\n{CAUSAL_GENERATION_SUFFIX}Topic:\n"
    )

_bad_prompt_endings = causal_prompt_df[
    ~causal_prompt_df["prompt"].map(
        lambda text: str(text).endswith(CAUSAL_EXACT_PROMPT_ENDING)
    )
]
if len(_bad_prompt_endings) != 0:
    raise RuntimeError(
        f"{len(_bad_prompt_endings)} causal prompts failed the exact-ending audit."
    )

print("Verified exact causal prompt ending:", repr(CAUSAL_EXACT_PROMPT_ENDING))
print("\nScreening original prompt tail:\n", repr(
    causal_screening_prompt_df.iloc[0]["prompt"][-700:]
))
print("\nHeld-out controlled prompt tail:\n", repr(
    causal_holdout_prompt_df[
        causal_holdout_prompt_df["condition"] == "controlled"
    ].iloc[0]["prompt"][-700:]
))

## 10. Select the causal features by activation-level AFR

The feature-selection sequence is deliberately two-stage:

1. **Filter:** original average preactivation is strictly largest on \(L\), and the feature passes the original-\(L\) firing-frequency threshold.
2. **Rank:** among those eligible features, rank by the pooled matched-pairs rank-biserial effect of activation under \(L\)-exclusion versus the matched original prompts.

No top-\(K\) selectivity ranking is applied before the AFR ranking. Consequently, every causally tested feature is both originally \(L\)-specific and among the strongest observed activation-level AFR features for that label.


In [ ]:
# ============================================================
# 18. Activation-level AFR discovery and causal feature selection
# ============================================================


def build_labeled_afr_controlled_prompt_set(
    removed_label: int,
) -> Tuple[List[str], List[Dict[str, Any]], torch.Tensor]:
    """
    Build the labeled, one-to-one-matched controlled prompts used only to rank
    activation-level AFR for one removed label.

    Returns
    -------
    controlled_prompts
    controlled_meta
    aligned_original_rows
        Original prompt rows corresponding exactly to the controlled prompts.
    """
    removed_label = int(removed_label)
    local_kept_labels = [label for label in ALL_LABELS if label != removed_label]

    controlled_prompts = []
    controlled_meta = []
    aligned_original_rows = []

    for original_row, original_item in enumerate(original_meta):
        final_label = int(original_item["final_label"])
        within_label_idx = int(original_item["within_label_idx"])

        if within_label_idx >= int(CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL):
            continue
        if final_label == removed_label:
            continue

        original_examples_8 = reconstruct_examples_from_meta(original_item)
        controlled_examples_8, replacements, shared_positions = (
            build_controlled_examples_for_removed_label(
                original_examples_8,
                pair_id=int(original_item["pair_id"]),
                removed_label=removed_label,
            )
        )
        controlled_prompt, controlled_span, first_sentence = (
            render_prompt_with_final_span(
                controlled_examples_8,
                allowed_labels=local_kept_labels,
            )
        )

        controlled_prompts.append(controlled_prompt)
        controlled_meta.append({
            "pair_id": int(original_item["pair_id"]),
            "within_label_idx": within_label_idx,
            "final_label": final_label,
            "final_label_name": LABEL_TO_WORD[final_label],
            "final_example_key": original_item["final_example_key"],
            "final_text_hash": original_item["final_text_hash"],
            "condition": f"activation_afr_without_{LABEL_TO_WORD[removed_label].lower()}",
            "removed_label": removed_label,
            "removed_label_name": LABEL_TO_WORD[removed_label],
            "allowed_labels": list(local_kept_labels),
            "instruction_first_sentence": first_sentence,
            "replacement_records": replacements,
            "shared_context_positions_1indexed": [
                int(position + 1) for position in shared_positions
            ],
            "final_char_span": tuple(map(int, controlled_span)),
        })
        aligned_original_rows.append(int(original_row))

    expected = 3 * int(CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL)
    if len(controlled_prompts) != expected:
        raise RuntimeError(
            f"Expected {expected} labeled AFR controlled prompts after removing "
            f"{LABEL_TO_WORD[removed_label]}, found {len(controlled_prompts)}."
        )

    aligned_original_rows = torch.tensor(aligned_original_rows, dtype=torch.long)
    controlled_df = pd.DataFrame(controlled_meta)
    observed_counts = controlled_df.groupby("final_label").size().to_dict()
    expected_counts = {
        int(label): int(CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL)
        for label in local_kept_labels
    }
    if observed_counts != expected_counts:
        raise RuntimeError(
            f"Controlled AFR label counts {observed_counts} != {expected_counts}."
        )

    return controlled_prompts, controlled_meta, aligned_original_rows


def paired_rank_biserial_only(diff_values: Sequence[float]) -> Dict[str, float]:
    """Paired rank-biserial effect without the extra Wilcoxon p-value pass."""
    diff = np.asarray(diff_values, dtype=np.float64)
    diff = diff[np.isfinite(diff)]
    nonzero = diff[diff != 0]

    if len(nonzero) == 0:
        return {
            "rank_biserial": 0.0,
            "n_total": int(len(diff)),
            "n_nonzero": 0,
        }

    ranks = rankdata(np.abs(nonzero), method="average")
    w_plus = float(ranks[nonzero > 0].sum())
    w_minus = float(ranks[nonzero < 0].sum())
    denominator = w_plus + w_minus
    rank_biserial = (
        (w_plus - w_minus) / denominator
        if denominator > 0
        else 0.0
    )
    return {
        "rank_biserial": float(rank_biserial),
        "n_total": int(len(diff)),
        "n_nonzero": int(len(nonzero)),
    }


def build_primary_label_eligibility_table(primary_label: int) -> pd.DataFrame:
    """
    Eligibility requires:
      1. original average PREACTIVATION strictly largest on primary_label;
      2. original primary-label firing frequency above the configured threshold.
    """
    primary_label = int(primary_label)
    other_labels = [label for label in ALL_LABELS if label != primary_label]

    pre_means = original_preactivation_label_mean.detach().float().cpu()
    firing = original_label_firing_frequency.detach().float().cpu()

    primary_mean = pre_means[primary_label]
    other_means = pre_means[other_labels]
    strongest_other_mean, strongest_other_local = other_means.max(dim=0)
    strongest_other_label = torch.tensor(other_labels, dtype=torch.long)[
        strongest_other_local
    ]
    preactivation_margin = primary_mean - strongest_other_mean
    preferred_label = pre_means.argmax(dim=0)
    primary_fire_frequency = firing[primary_label]

    eligible_mask = (
        (preferred_label == primary_label)
        & (preactivation_margin > 0)
        & (
            primary_fire_frequency
            >= float(CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY)
        )
    )
    feature_indices = torch.where(eligible_mask)[0]

    return pd.DataFrame({
        "primary_label": primary_label,
        "primary_label_name": LABEL_TO_WORD[primary_label],
        "feature_idx": feature_indices.numpy().astype(int),
        "primary_original_mean_preactivation": primary_mean[
            feature_indices
        ].numpy(),
        "strongest_other_label": strongest_other_label[
            feature_indices
        ].numpy().astype(int),
        "strongest_other_label_name": [
            LABEL_TO_WORD[int(label)]
            for label in strongest_other_label[feature_indices]
        ],
        "strongest_other_original_mean_preactivation": strongest_other_mean[
            feature_indices
        ].numpy(),
        "preactivation_selectivity_margin": preactivation_margin[
            feature_indices
        ].numpy(),
        "primary_firing_frequency": primary_fire_frequency[
            feature_indices
        ].numpy(),
        "firing_frequency_threshold": float(
            CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY
        ),
    })


def compute_activation_afr_candidates_for_label(
    removed_label: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[Dict[str, Any]]]:
    """Filter by preactivation/frequency, then rank by paired activation AFR."""
    removed_label = int(removed_label)
    eligibility_df = build_primary_label_eligibility_table(removed_label)
    if eligibility_df.empty:
        raise RuntimeError(
            f"No features satisfied preactivation specificity and firing frequency "
            f"for {LABEL_TO_WORD[removed_label]}."
        )

    controlled_prompts, controlled_meta, aligned_original_rows = (
        build_labeled_afr_controlled_prompt_set(removed_label)
    )

    print(
        f"Computing activation-level AFR for {LABEL_TO_WORD[removed_label]}: "
        f"{len(eligibility_df)} eligible features, "
        f"{len(controlled_prompts)} paired retained-label prompts..."
    )
    controlled_scores, controlled_fired = extract_example_level_scores(
        controlled_prompts,
        controlled_meta,
        batch_size=int(CAUSAL_AFR_DISCOVERY_BATCH_SIZE),
        activation_kind=CAUSAL_AFR_RANK_ACTIVATION_TYPE,
        return_both=False,
        verbose=False,
    )

    original_source = (
        example_level_preactivation_original
        if CAUSAL_AFR_RANK_ACTIVATION_TYPE == "pre"
        else example_level_postactivation_original
    )
    original_aligned = original_source[aligned_original_rows]
    if original_aligned.shape != controlled_scores.shape:
        raise RuntimeError(
            f"AFR aligned shape mismatch: {tuple(original_aligned.shape)} vs "
            f"{tuple(controlled_scores.shape)}"
        )

    candidate_features = eligibility_df["feature_idx"].to_numpy(dtype=np.int64)
    diff_matrix = (
        controlled_scores[:, candidate_features].detach().float().cpu()
        - original_aligned[:, candidate_features].detach().float().cpu()
    )

    mean_shift = diff_matrix.mean(dim=0).numpy()
    median_shift = diff_matrix.median(dim=0).values.numpy()
    fraction_positive = (diff_matrix > 0).float().mean(dim=0).numpy()

    rb_values = np.zeros(len(candidate_features), dtype=np.float64)
    n_nonzero = np.zeros(len(candidate_features), dtype=np.int64)
    for local_idx in range(len(candidate_features)):
        stats = paired_rank_biserial_only(diff_matrix[:, local_idx].numpy())
        rb_values[local_idx] = stats["rank_biserial"]
        n_nonzero[local_idx] = stats["n_nonzero"]

    afr_df = eligibility_df.copy()
    afr_df["activation_afr_type"] = CAUSAL_AFR_RANK_ACTIVATION_TYPE
    afr_df["activation_afr_rank_biserial"] = rb_values
    afr_df["activation_afr_mean_shift"] = mean_shift
    afr_df["activation_afr_median_shift"] = median_shift
    afr_df["activation_afr_fraction_positive"] = fraction_positive
    afr_df["activation_afr_n_pairs"] = int(diff_matrix.shape[0])
    afr_df["activation_afr_n_nonzero"] = n_nonzero

    ranking_pool = afr_df.copy()
    if CAUSAL_REQUIRE_POSITIVE_AFR:
        ranking_pool = ranking_pool[
            ranking_pool["activation_afr_rank_biserial"] > 0
        ].copy()

    ranking_pool = ranking_pool.sort_values(
        [
            "activation_afr_rank_biserial",
            "activation_afr_mean_shift",
            "preactivation_selectivity_margin",
        ],
        ascending=False,
    ).reset_index(drop=True)

    if len(ranking_pool) < int(CAUSAL_TOP_FEATURES_PER_LABEL):
        raise RuntimeError(
            f"Only {len(ranking_pool)} eligible positive-AFR features for "
            f"{LABEL_TO_WORD[removed_label]}; need "
            f"{CAUSAL_TOP_FEATURES_PER_LABEL}."
        )

    selected = ranking_pool.head(int(CAUSAL_TOP_FEATURES_PER_LABEL)).copy()
    selected.insert(2, "feature_rank", np.arange(1, len(selected) + 1))
    selected["selection_rule"] = (
        "preactivation_primary_and_frequency_then_activation_AFR_rank_biserial"
    )
    selected["alpha_mode"] = "constant"
    selected["steering_alpha"] = float(CAUSAL_ALPHA)

    decoder_norms = []
    for feature_idx in selected["feature_idx"].astype(int):
        decoder_norms.append(
            float(sae.W_dec[feature_idx].detach().float().norm().cpu())
        )
    selected["decoder_norm"] = decoder_norms
    selected["residual_shift_norm_per_token"] = (
        abs(float(CAUSAL_ALPHA)) * selected["decoder_norm"]
    )

    del controlled_scores, controlled_fired, original_aligned, diff_matrix
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return afr_df, selected, controlled_meta


_causal_afr_candidate_tables = []
_causal_selected_tables = []
causal_activation_afr_meta_by_removed_label = {}

for _removed_label in CAUSAL_REMOVED_LABELS:
    _removed_label = int(_removed_label)
    _candidate_df, _selected_df, _controlled_meta = (
        compute_activation_afr_candidates_for_label(_removed_label)
    )
    _causal_afr_candidate_tables.append(_candidate_df)
    _causal_selected_tables.append(_selected_df)
    causal_activation_afr_meta_by_removed_label[_removed_label] = _controlled_meta

causal_activation_afr_candidate_df = pd.concat(
    _causal_afr_candidate_tables,
    ignore_index=True,
).sort_values(
    ["primary_label", "activation_afr_rank_biserial"],
    ascending=[True, False],
).reset_index(drop=True)

causal_selected_feature_df = pd.concat(
    _causal_selected_tables,
    ignore_index=True,
).sort_values(
    ["primary_label", "feature_rank"]
).reset_index(drop=True)

# Final defensive checks: every selected feature must satisfy both eligibility
# criteria, and the selected order must be descending in activation-level AFR.
if not (
    causal_selected_feature_df["preactivation_selectivity_margin"] > 0
).all():
    raise AssertionError("A selected causal feature failed preactivation specificity.")
if not (
    causal_selected_feature_df["primary_firing_frequency"]
    + 1e-12
    >= float(CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY)
).all():
    raise AssertionError("A selected causal feature failed the firing-frequency filter.")
if CAUSAL_REQUIRE_POSITIVE_AFR and not (
    causal_selected_feature_df["activation_afr_rank_biserial"] > 0
).all():
    raise AssertionError("A selected causal feature did not have positive activation AFR.")

print("Causal AFR configuration")
print("  removed labels:", [LABEL_TO_WORD[int(x)] for x in CAUSAL_REMOVED_LABELS])
print("  eligibility specificity: original mean PREACTIVATION strictly strongest on L")
print("  minimum original-L firing frequency:", CAUSAL_MIN_PRIMARY_FIRING_FREQUENCY)
print("  AFR ranking activation type:", CAUSAL_AFR_RANK_ACTIVATION_TYPE)
print("  AFR discovery pairs per removed label:", 3 * int(CAUSAL_AFR_DISCOVERY_EXAMPLES_PER_LABEL))
print("  top features per label:", CAUSAL_TOP_FEATURES_PER_LABEL)
print("  causal-screen original prompts per label group:", 4 * int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL))
print("  held-out original prompts per label group:", 4 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL))
print("  held-out controlled prompts per label group:", 3 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL))
print("  constant alpha:", CAUSAL_ALPHA)
print("  token scope:", CAUSAL_TOKEN_SCOPE)
if CAUSAL_TOKEN_SCOPE == "last_k":
    print("  causal last-k offsets before label:", CAUSAL_LAST_K)
print("  normalize decoder direction:", CAUSAL_NORMALIZE_DECODER_DIRECTION)
print("  causal batch size:", CAUSAL_BATCH_SIZE)

print("Selected activation-AFR features")
display(causal_selected_feature_df)

del _causal_afr_candidate_tables, _causal_selected_tables
try:
    del _candidate_df, _selected_df, _controlled_meta, _removed_label
except NameError:
    pass


In [ ]:
# ============================================================
# 19. Batched next-label probability scoring and feature steering
# ============================================================


def infer_exact_single_token_label_ids(
    representative_prompts: Sequence[str],
) -> Dict[int, int]:
    """
    Infer the exact next-token ID for each AGNews label at the configured causal prompt boundary.

    The fast causal scorer requires each label completion to be one token and
    requires those token IDs to be unique. The function verifies this at both an
    original and controlled representative prompt boundary.
    """
    inferred_by_prompt = []

    for prompt in representative_prompts:
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
        local = {}

        for label_id, label_word in LABEL_TO_WORD.items():
            completion = CAUSAL_LABEL_COMPLETION_PREFIX + str(label_word)
            full_ids = tokenizer.encode(
                prompt + completion,
                add_special_tokens=False,
            )

            if full_ids[: len(prompt_ids)] != prompt_ids:
                raise RuntimeError(
                    "Appending a candidate label retokenized the causal prompt "
                    f"boundary for label {label_word!r}."
                )

            suffix = full_ids[len(prompt_ids):]
            if len(suffix) != 1:
                raise RuntimeError(
                    f"AGNews label {label_word!r} is not a single token at the "
                    f"causal prompt boundary; suffix IDs={suffix}."
                )
            local[int(label_id)] = int(suffix[0])

        if len(set(local.values())) != len(local):
            raise RuntimeError(f"Candidate label token IDs are not unique: {local}")
        inferred_by_prompt.append(local)

    first = inferred_by_prompt[0]
    if any(mapping != first for mapping in inferred_by_prompt[1:]):
        raise RuntimeError(
            f"Label token IDs differ across representative prompts: {inferred_by_prompt}"
        )
    return first


_representative_prompts = [
    causal_holdout_prompt_df[causal_holdout_prompt_df["condition"] == "original"].iloc[0]["prompt"],
    causal_holdout_prompt_df[causal_holdout_prompt_df["condition"] == "controlled"].iloc[0]["prompt"],
]
CAUSAL_LABEL_TOKEN_IDS = infer_exact_single_token_label_ids(_representative_prompts)
CAUSAL_LABEL_ID_ORDER = list(ALL_LABELS)
CAUSAL_LABEL_TOKEN_ID_LIST = [
    CAUSAL_LABEL_TOKEN_IDS[label] for label in CAUSAL_LABEL_ID_ORDER
]

print("Verified exact one-token AGNews label completions:")
display(pd.DataFrame({
    "label_id": CAUSAL_LABEL_ID_ORDER,
    "label_name": [LABEL_TO_WORD[label] for label in CAUSAL_LABEL_ID_ORDER],
    "token_id": CAUSAL_LABEL_TOKEN_ID_LIST,
    "decoded_token": [
        repr(tokenizer.decode([token_id]))
        for token_id in CAUSAL_LABEL_TOKEN_ID_LIST
    ],
}))


def causal_query_token_indices(
    offset_mapping: Sequence[Tuple[int, int]],
    input_ids_1d,
    query_char_span: Tuple[int, int],
    article_char_span: Tuple[int, int],
    attention_mask_1d=None,
) -> List[int]:
    """
    Return the prompt positions edited by the causal intervention.

    For ``CAUSAL_TOKEN_SCOPE == "last_k"``, offsets are defined relative to the
    end of the complete *unlabeled* causal prefix, not relative to the labeled
    activation-extraction span:

      - ``-1`` is the final real input token immediately before the model predicts
        the first label token;
      - ``-2`` is the preceding real input token; and so on.

    The candidate label is never included in ``input_ids_1d``. Therefore all
    selected positions necessarily precede label generation.
    """
    seq_len = int(len(input_ids_1d))

    if CAUSAL_TOKEN_SCOPE == "last_k":
        if attention_mask_1d is None:
            real_positions = list(range(seq_len))
        else:
            mask = torch.as_tensor(attention_mask_1d).detach().cpu().bool().reshape(-1)
            if int(mask.numel()) != seq_len:
                raise ValueError(
                    f"attention_mask length {mask.numel()} != sequence length {seq_len}."
                )
            real_positions = torch.where(mask)[0].tolist()

        if len(real_positions) == 0:
            raise RuntimeError("The causal prompt contains no real (unmasked) tokens.")

        indices = []
        for offset in CAUSAL_LAST_K:
            offset = int(offset)
            if abs(offset) > len(real_positions):
                raise IndexError(
                    f"CAUSAL_LAST_K offset {offset} is out of range for a prompt "
                    f"with {len(real_positions)} real tokens."
                )
            indices.append(int(real_positions[offset]))

        # Preserve left-to-right sequence order even when the user writes
        # offsets as [-1, -2].
        indices = sorted(indices)

    else:
        query_tok_start, query_tok_end = char_span_to_token_span(
            offset_mapping,
            int(query_char_span[0]),
            int(query_char_span[1]),
        )

        if CAUSAL_TOKEN_SCOPE == "article":
            article_tok_start, article_tok_end = char_span_to_token_span(
                offset_mapping,
                int(article_char_span[0]),
                int(article_char_span[1]),
            )
            indices = list(range(article_tok_start, article_tok_end))
        elif CAUSAL_TOKEN_SCOPE == "full_query":
            indices = list(range(query_tok_start, query_tok_end))
        else:
            raise ValueError(CAUSAL_TOKEN_SCOPE)

    if len(indices) == 0:
        raise RuntimeError("Causal intervention token selection is empty.")
    if len(set(indices)) != len(indices):
        raise RuntimeError(f"Duplicate causal intervention positions: {indices}")
    return [int(index) for index in indices]


def debug_causal_intervention_tokens(row_index: int = 0) -> pd.DataFrame:
    """Show the exact prompt tokens edited by the current causal token scope."""
    row = causal_holdout_prompt_df.iloc[int(row_index)]
    prompt = row["prompt"]
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )
    input_ids = encoded["input_ids"][0]
    attention_mask = encoded["attention_mask"][0]
    offsets = encoded["offset_mapping"][0].tolist()

    selected = causal_query_token_indices(
        offsets,
        input_ids,
        row["query_char_span"],
        row["article_char_span"],
        attention_mask_1d=attention_mask,
    )
    selected_set = set(selected)

    real_positions = torch.where(attention_mask.detach().cpu().bool())[0].tolist()
    end_offset_by_position = {
        int(position): int(local_idx - len(real_positions))
        for local_idx, position in enumerate(real_positions)
    }

    if CAUSAL_TOKEN_SCOPE == "last_k":
        show_start = max(0, min(selected) - 10)
        show_end = min(len(input_ids), max(selected) + 3)
    else:
        query_start, query_end = char_span_to_token_span(
            offsets,
            *row["query_char_span"],
        )
        show_start = max(0, query_start - 4)
        show_end = min(len(input_ids), query_end + 8)

    rows = []
    for token_idx in range(show_start, show_end):
        char_start, char_end = offsets[token_idx]
        rows.append({
            "token_idx": int(token_idx),
            "offset_before_first_label_token": end_offset_by_position.get(
                int(token_idx), np.nan
            ),
            "selected_for_steering": bool(token_idx in selected_set),
            "token_id": int(input_ids[token_idx]),
            "decoded_token": repr(tokenizer.decode([int(input_ids[token_idx])])),
            "text_piece": (
                repr(prompt[char_start:char_end])
                if char_end > char_start
                else "''"
            ),
            "char_start": int(char_start),
            "char_end": int(char_end),
        })

    result = pd.DataFrame(rows)

    if CAUSAL_TOKEN_SCOPE == "last_k":
        observed_offsets = sorted(
            int(end_offset_by_position[position]) for position in selected
        )
        expected_offsets = sorted(map(int, CAUSAL_LAST_K))
        if observed_offsets != expected_offsets:
            raise AssertionError(
                f"Selected causal offsets {observed_offsets} != configured "
                f"CAUSAL_LAST_K {expected_offsets}."
            )

    return result


print("Causal token-span sanity check:")
_causal_debug_df = debug_causal_intervention_tokens(0)
display(_causal_debug_df)
if CAUSAL_TOKEN_SCOPE == "last_k":
    print(
        "Verified: CAUSAL_LAST_K is measured from the complete unlabeled prompt "
        "boundary. The label token is not in the input; it is scored as the next token."
    )
del _causal_debug_df


def make_batched_feature_steering_hook(
    feature_indices: Sequence[int],
    steering_alphas: Sequence[float],
    token_mask: torch.Tensor,
):
    feature_indices = [int(feature_idx) for feature_idx in feature_indices]
    steering_alphas = [float(alpha) for alpha in steering_alphas]

    if len(feature_indices) != len(steering_alphas):
        raise ValueError("feature_indices and steering_alphas must have equal length")
    if token_mask.shape[0] != len(feature_indices):
        raise ValueError(
            f"token_mask batch {token_mask.shape[0]} != feature batch {len(feature_indices)}"
        )

    feature_tensor = torch.tensor(
        feature_indices,
        dtype=torch.long,
        device=sae.device,
    )
    directions = sae.W_dec.index_select(0, feature_tensor).detach().float()
    if CAUSAL_NORMALIZE_DECODER_DIRECTION:
        directions = directions / directions.norm(dim=-1, keepdim=True).clamp_min(1e-9)
    alpha_tensor = torch.tensor(
        steering_alphas,
        dtype=torch.float32,
        device=directions.device,
    )

    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.shape[0] != len(feature_indices):
            raise RuntimeError(
                f"Hook batch {hidden.shape[0]} != steering batch {len(feature_indices)}"
            )

        edited = hidden.clone()
        local_mask = token_mask.to(device=edited.device, dtype=torch.bool)
        local_directions = directions.to(device=edited.device, dtype=edited.dtype)
        local_alphas = alpha_tensor.to(device=edited.device, dtype=edited.dtype)

        for batch_idx in range(edited.shape[0]):
            positions = torch.where(local_mask[batch_idx])[0]
            if positions.numel() == 0:
                raise RuntimeError(f"Empty steering positions for batch row {batch_idx}")
            delta = local_alphas[batch_idx] * local_directions[batch_idx]
            edited[batch_idx, positions, :] = (
                edited[batch_idx, positions, :] + delta
            )

        if isinstance(output, tuple):
            return (edited,) + tuple(output[1:])
        return edited

    return hook


def forward_with_last_token_logits(input_ids, attention_mask):
    """Request only final-position logits when supported by this Transformers version."""
    kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "use_cache": False,
    }
    parameters = inspect.signature(model.forward).parameters
    if "logits_to_keep" in parameters:
        kwargs["logits_to_keep"] = 1
    elif "num_logits_to_keep" in parameters:
        kwargs["num_logits_to_keep"] = 1

    output = model(**kwargs)
    return output.logits[:, -1, :].float()


@torch.no_grad()
def score_causal_prompt_batch(
    rows: Sequence[Dict[str, Any]],
    feature_indices: Optional[Sequence[int]] = None,
    steering_alphas: Optional[Sequence[float]] = None,
) -> np.ndarray:
    """
    Return [batch, 4] probabilities normalized over the four AGNews labels.

    When feature_indices is supplied, each batch row receives its own feature
    direction and alpha on the selected final-query tokens.
    """
    rows = list(rows)
    if len(rows) == 0:
        return np.zeros((0, len(ALL_LABELS)), dtype=np.float64)

    prompts = [str(row["prompt"]) for row in rows]

    # Left padding ensures the last sequence position is the final real token
    # of the configured causal prefix for every row, including when
    # last-logit-only output is used.
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    try:
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,
            add_special_tokens=False,
            return_offsets_mapping=True,
        )
    finally:
        tokenizer.padding_side = old_padding_side

    offset_mapping = encoded.pop("offset_mapping")
    input_ids = encoded["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = encoded["attention_mask"].to(MODEL_INPUT_DEVICE)

    handle = None
    if feature_indices is not None:
        if steering_alphas is None:
            raise ValueError("steering_alphas is required when feature_indices is supplied")
        if len(feature_indices) != len(rows) or len(steering_alphas) != len(rows):
            raise ValueError("One feature index and alpha are required per prompt row")

        token_mask = torch.zeros(
            (len(rows), input_ids.shape[1]),
            dtype=torch.bool,
        )
        for batch_idx, row in enumerate(rows):
            offsets = offset_mapping[batch_idx].tolist()
            input_ids_cpu = input_ids[batch_idx].detach().cpu()
            attention_mask_cpu = attention_mask[batch_idx].detach().cpu()
            selected = causal_query_token_indices(
                offsets,
                input_ids_cpu,
                row["query_char_span"],
                row["article_char_span"],
                attention_mask_1d=attention_mask_cpu,
            )
            token_mask[batch_idx, selected] = True

        hook = make_batched_feature_steering_hook(
            feature_indices=feature_indices,
            steering_alphas=steering_alphas,
            token_mask=token_mask,
        )
        block = model.get_submodule(f"model.layers.{int(TARGET_LAYER)}")
        handle = block.register_forward_hook(hook)

    try:
        final_logits = forward_with_last_token_logits(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
    finally:
        if handle is not None:
            handle.remove()

    label_token_tensor = torch.tensor(
        CAUSAL_LABEL_TOKEN_ID_LIST,
        dtype=torch.long,
        device=final_logits.device,
    )
    label_logits = final_logits.index_select(dim=-1, index=label_token_tensor)
    label_probabilities = torch.softmax(label_logits, dim=-1)

    result = label_probabilities.detach().float().cpu().numpy().astype(np.float64)

    del final_logits, input_ids, attention_mask
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


def score_unique_causal_baselines(prompt_table: pd.DataFrame) -> pd.DataFrame:
    """Score every unique prompt once and return its four-label baseline probabilities."""
    unique = (
        prompt_table.drop_duplicates("prompt_hash")
        [["prompt_hash", "prompt", "query_char_span", "article_char_span"]]
        .reset_index(drop=True)
    )

    output_rows = []
    n_batches = math.ceil(len(unique) / int(CAUSAL_BATCH_SIZE))

    for batch_number, start in enumerate(
        range(0, len(unique), int(CAUSAL_BATCH_SIZE)),
        start=1,
    ):
        batch_df = unique.iloc[start:start + int(CAUSAL_BATCH_SIZE)]
        rows = batch_df.to_dict("records")
        probabilities = score_causal_prompt_batch(rows)

        for local_idx, row in enumerate(rows):
            record = {"prompt_hash": row["prompt_hash"]}
            for column_idx, label_id in enumerate(CAUSAL_LABEL_ID_ORDER):
                record[f"baseline_prob_{LABEL_TO_WORD[label_id]}"] = float(
                    probabilities[local_idx, column_idx]
                )
            record["baseline_pred_label"] = int(
                CAUSAL_LABEL_ID_ORDER[int(np.argmax(probabilities[local_idx]))]
            )
            output_rows.append(record)


    return pd.DataFrame(output_rows)


In [ ]:
# ============================================================
# 20. Two-stage causal experiment: screening, then held-out evaluation
# ============================================================

from IPython.display import clear_output


def _merge_causal_baselines(prompt_table: pd.DataFrame, stage_name: str) -> pd.DataFrame:
    unique_count = int(prompt_table["prompt_hash"].nunique())
    print(
        f"Scoring {unique_count} unique {stage_name} baseline prompts "
        f"with batch size {int(CAUSAL_BATCH_SIZE)}..."
    )
    baseline_df = score_unique_causal_baselines(prompt_table)
    scored = prompt_table.merge(
        baseline_df,
        on="prompt_hash",
        how="left",
        validate="many_to_one",
    )
    baseline_columns = [
        f"baseline_prob_{LABEL_TO_WORD[label]}"
        for label in CAUSAL_LABEL_ID_ORDER
    ]
    if scored[baseline_columns].isna().any().any():
        raise RuntimeError(f"Missing {stage_name} baseline probabilities after merge.")
    return scored


def _score_one_feature_on_scored_prompts(
    scored_prompt_table: pd.DataFrame,
    selected,
    *,
    conditions: Sequence[str],
) -> pd.DataFrame:
    """Run one selected feature on the requested conditions and return long CE rows."""
    removed_label = int(selected.primary_label)
    feature_idx = int(selected.feature_idx)
    feature_rank = int(selected.feature_rank)
    alpha = float(CAUSAL_ALPHA)

    baseline_columns = [
        f"baseline_prob_{LABEL_TO_WORD[label]}"
        for label in CAUSAL_LABEL_ID_ORDER
    ]
    causal_rows = []

    for condition in conditions:
        condition_df = scored_prompt_table[
            (scored_prompt_table["removed_label"] == removed_label)
            & (scored_prompt_table["condition"] == condition)
        ].reset_index(drop=True)
        if condition_df.empty:
            raise RuntimeError(
                f"No {condition!r} prompts for removed label {removed_label}."
            )
        if condition == "controlled" and (
            condition_df["gold_label"] == removed_label
        ).any():
            raise AssertionError("Controlled prompts contain the removed gold label.")

        all_steered = []
        for start in range(0, len(condition_df), int(CAUSAL_BATCH_SIZE)):
            batch_df = condition_df.iloc[start:start + int(CAUSAL_BATCH_SIZE)]
            batch_rows = batch_df.to_dict("records")
            all_steered.append(
                score_causal_prompt_batch(
                    batch_rows,
                    feature_indices=[feature_idx] * len(batch_rows),
                    steering_alphas=[alpha] * len(batch_rows),
                )
            )

        steered_probabilities = np.concatenate(all_steered, axis=0)
        baseline_probability_matrix = condition_df[baseline_columns].to_numpy(
            dtype=np.float64
        )
        if steered_probabilities.shape != baseline_probability_matrix.shape:
            raise RuntimeError(
                f"Steered/baseline shape mismatch: {steered_probabilities.shape} "
                f"vs {baseline_probability_matrix.shape}."
            )
        delta_probability_matrix = (
            steered_probabilities - baseline_probability_matrix
        )

        for prompt_local_idx, prompt_row in condition_df.iterrows():
            for candidate_col, candidate_label in enumerate(CAUSAL_LABEL_ID_ORDER):
                causal_rows.append({
                    "causal_split": str(prompt_row["causal_split"]),
                    "removed_label": removed_label,
                    "removed_label_name": LABEL_TO_WORD[removed_label],
                    "feature_rank": feature_rank,
                    "feature_idx": feature_idx,
                    "steering_alpha": alpha,
                    "condition": condition,
                    "causal_prompt_id": int(prompt_row["causal_prompt_id"]),
                    "pair_id": int(prompt_row["pair_id"]),
                    "within_label_idx": int(prompt_row["within_label_idx"]),
                    "final_example_key": str(prompt_row["final_example_key"]),
                    "gold_label": int(prompt_row["gold_label"]),
                    "gold_label_name": str(prompt_row["gold_label_name"]),
                    "candidate_label": int(candidate_label),
                    "candidate_label_name": LABEL_TO_WORD[int(candidate_label)],
                    "baseline_probability": float(
                        baseline_probability_matrix[prompt_local_idx, candidate_col]
                    ),
                    "steered_probability": float(
                        steered_probabilities[prompt_local_idx, candidate_col]
                    ),
                    "delta_probability": float(
                        delta_probability_matrix[prompt_local_idx, candidate_col]
                    ),
                })

    return pd.DataFrame(causal_rows)


def _screening_summary_for_feature(
    feature_long_df: pd.DataFrame,
    selected,
) -> Dict[str, Any]:
    """Compute green/red on the independent original-prompt screening set."""
    if set(feature_long_df["condition"]) != {"original"}:
        raise RuntimeError("Causal screening must use original prompts only.")

    ce = (
        feature_long_df
        .groupby([
            "removed_label",
            "removed_label_name",
            "feature_rank",
            "feature_idx",
            "steering_alpha",
            "candidate_label",
            "candidate_label_name",
        ], as_index=False)
        .agg(
            mean_causal_effect=("delta_probability", "mean"),
            fraction_effect_positive=(
                "delta_probability",
                lambda values: float(np.mean(np.asarray(values) > 0)),
            ),
            n_prompts=("delta_probability", "count"),
        )
    )

    removed_label = int(selected.primary_label)
    feature_idx = int(selected.feature_idx)
    expected = 4 * int(CAUSAL_SCREENING_EXAMPLES_PER_LABEL)
    if set(ce["n_prompts"].astype(int)) != {expected}:
        raise RuntimeError(
            f"Screening CE for feature {feature_idx} must use {expected} prompts."
        )

    primary = ce[ce["candidate_label"] == removed_label]
    nonprimary = ce[ce["candidate_label"] != removed_label]
    if len(primary) != 1 or len(nonprimary) != 3:
        raise RuntimeError("Invalid screening primary/non-primary CE rows.")

    primary_row = primary.iloc[0]
    best_nonprimary = nonprimary.loc[nonprimary["mean_causal_effect"].idxmax()]
    primary_effect = float(primary_row["mean_causal_effect"])
    nonprimary_effect = float(best_nonprimary["mean_causal_effect"])
    causal_margin = primary_effect - nonprimary_effect

    pass_primary_positive = bool(primary_effect > 0)
    pass_primary_gt_nonprimary = bool(primary_effect > nonprimary_effect)
    causal_screen_pass = bool(
        (pass_primary_positive or not CAUSAL_SCREEN_REQUIRE_PRIMARY_POSITIVE)
        and (
            pass_primary_gt_nonprimary
            or not CAUSAL_SCREEN_REQUIRE_PRIMARY_GT_NONPRIMARY
        )
    )

    return {
        "removed_label": removed_label,
        "removed_label_name": LABEL_TO_WORD[removed_label],
        "feature_rank": int(selected.feature_rank),
        "feature_idx": feature_idx,
        "steering_alpha": float(CAUSAL_ALPHA),
        "activation_afr_rank_biserial": float(
            selected.activation_afr_rank_biserial
        ),
        "activation_afr_mean_shift": float(selected.activation_afr_mean_shift),
        "screen_original_primary_effect": primary_effect,
        "screen_original_strongest_nonprimary_label": int(
            best_nonprimary["candidate_label"]
        ),
        "screen_original_strongest_nonprimary_label_name": str(
            best_nonprimary["candidate_label_name"]
        ),
        "screen_original_strongest_nonprimary_effect": nonprimary_effect,
        "screen_causal_selectivity_margin": causal_margin,
        "screen_primary_positive": pass_primary_positive,
        "screen_primary_gt_nonprimary": pass_primary_gt_nonprimary,
        "causal_screen_pass": causal_screen_pass,
        "n_screen_original_prompts": expected,
    }


def _screening_dashboard_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    out = summary_df.copy().sort_values(["removed_label", "feature_rank"])
    out["screen_primary_CE_pp"] = 100.0 * out["screen_original_primary_effect"]
    out["screen_nonprimary_CE_pp"] = (
        100.0 * out["screen_original_strongest_nonprimary_effect"]
    )
    out["screen_primary_minus_nonprimary_pp"] = (
        100.0 * out["screen_causal_selectivity_margin"]
    )
    return out[[
        "removed_label_name",
        "feature_rank",
        "feature_idx",
        "activation_afr_rank_biserial",
        "screen_original_strongest_nonprimary_label_name",
        "screen_nonprimary_CE_pp",
        "screen_primary_CE_pp",
        "screen_primary_minus_nonprimary_pp",
        "screen_primary_positive",
        "screen_primary_gt_nonprimary",
        "causal_screen_pass",
    ]].rename(columns={
        "removed_label_name": "primary label",
        "feature_rank": "AFR rank",
        "feature_idx": "feature",
        "activation_afr_rank_biserial": "activation AFR r_rb",
        "screen_original_strongest_nonprimary_label_name": "best non-primary",
        "screen_primary_positive": "green_screen > 0",
        "screen_primary_gt_nonprimary": "green_screen > red_screen",
        "causal_screen_pass": "included in held-out figure",
    }).round(4)


def _screening_count_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for label in CAUSAL_REMOVED_LABELS:
        label = int(label)
        sub = summary_df[summary_df["removed_label"] == label]
        rows.append({
            "primary_label": LABEL_TO_WORD[label],
            "tested": int(len(sub)),
            "passed": int(sub["causal_screen_pass"].sum()),
            "pass_fraction": float(sub["causal_screen_pass"].mean()),
        })
    rows.append({
        "primary_label": "ALL",
        "tested": int(len(summary_df)),
        "passed": int(summary_df["causal_screen_pass"].sum()),
        "pass_fraction": float(summary_df["causal_screen_pass"].mean()),
    })
    return pd.DataFrame(rows)


def run_causal_specificity_screen(
    prompt_table: pd.DataFrame,
    feature_table: pd.DataFrame,
):
    """Screen all 40 features one at a time on independent original prompts."""
    if set(prompt_table["causal_split"]) != {"screening"}:
        raise ValueError("Screening prompt table has the wrong causal_split.")
    if set(prompt_table["condition"]) != {"original"}:
        raise ValueError("Screening prompt table must contain original prompts only.")

    scored = _merge_causal_baselines(prompt_table, "causal-screening")
    long_parts = []
    summaries = []
    total_features = int(len(feature_table))

    for feature_number, selected in enumerate(
        feature_table.itertuples(index=False),
        start=1,
    ):
        feature_long = _score_one_feature_on_scored_prompts(
            scored,
            selected,
            conditions=["original"],
        )
        long_parts.append(feature_long)
        summaries.append(_screening_summary_for_feature(feature_long, selected))

        summary_so_far = pd.DataFrame(summaries).sort_values(
            ["removed_label", "feature_rank"]
        ).reset_index(drop=True)
        globals()["causal_screening_feature_summary_so_far_df"] = (
            summary_so_far.copy()
        )

        if CAUSAL_SHOW_SCREENING_RESULTS:
            if CAUSAL_CLEAR_OUTPUT_BETWEEN_FEATURES:
                clear_output(wait=True)
            latest = summary_so_far.iloc[-1]
            print("=" * 112)
            print(
                f"Causal-specificity screening: {feature_number}/{total_features} | "
                f"label={latest['removed_label_name']} | "
                f"feature={int(latest['feature_idx'])} | "
                f"green-red={100.0 * float(latest['screen_causal_selectivity_margin']):+.3f} pp | "
                f"PASS={bool(latest['causal_screen_pass'])}"
            )
            print("=" * 112)
            display(_screening_dashboard_table(summary_so_far))
            print("Screening counts so far")
            display(_screening_count_table(summary_so_far))

    screening_long_df = pd.concat(long_parts, ignore_index=True)
    screening_summary_df = pd.DataFrame(summaries).sort_values(
        ["removed_label", "feature_rank"]
    ).reset_index(drop=True)

    screen_merge_keys = [
        "removed_label", "feature_rank", "feature_idx", "steering_alpha"
    ]
    screen_result_columns = [
        "screen_original_primary_effect",
        "screen_original_strongest_nonprimary_label",
        "screen_original_strongest_nonprimary_label_name",
        "screen_original_strongest_nonprimary_effect",
        "screen_causal_selectivity_margin",
        "screen_primary_positive",
        "screen_primary_gt_nonprimary",
        "causal_screen_pass",
        "n_screen_original_prompts",
    ]
    screen_for_merge = screening_summary_df[
        screen_merge_keys + screen_result_columns
    ].rename(columns={"removed_label": "primary_label"})
    final_feature_table = feature_table.merge(
        screen_for_merge,
        on=["primary_label", "feature_rank", "feature_idx", "steering_alpha"],
        how="left",
        validate="one_to_one",
    )
    if final_feature_table["causal_screen_pass"].isna().any():
        raise RuntimeError("Missing causal-screening results after selection merge.")
    final_feature_table = final_feature_table[
        final_feature_table["causal_screen_pass"]
    ].copy().sort_values(["primary_label", "feature_rank"]).reset_index(drop=True)
    final_feature_table["final_selection_rule"] = (
        "activation_AFR_top10_then_independent_screen_primary_positive_and_"
        "primary_gt_strongest_nonprimary"
    )

    return scored, screening_long_df, screening_summary_df, final_feature_table


def _heldout_feature_summary(
    feature_long_df: pd.DataFrame,
    selected,
) -> Dict[str, Any]:
    """Summarize held-out red/blue/green effects for one screened feature."""
    ce = (
        feature_long_df
        .groupby([
            "removed_label",
            "removed_label_name",
            "feature_rank",
            "feature_idx",
            "steering_alpha",
            "condition",
            "candidate_label",
            "candidate_label_name",
        ], as_index=False)
        .agg(
            mean_causal_effect=("delta_probability", "mean"),
            fraction_effect_positive=(
                "delta_probability",
                lambda values: float(np.mean(np.asarray(values) > 0)),
            ),
            n_prompts=("delta_probability", "count"),
        )
    )

    removed_label = int(selected.primary_label)
    feature_idx = int(selected.feature_idx)
    original = ce[ce["condition"] == "original"]
    controlled = ce[ce["condition"] == "controlled"]
    expected_original = 4 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
    expected_controlled = 3 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
    if set(original["n_prompts"].astype(int)) != {expected_original}:
        raise RuntimeError("Held-out original prompt count mismatch.")
    if set(controlled["n_prompts"].astype(int)) != {expected_controlled}:
        raise RuntimeError("Held-out controlled prompt count mismatch.")

    primary = original[original["candidate_label"] == removed_label]
    original_nonprimary = original[original["candidate_label"] != removed_label]
    controlled_nonprimary = controlled[
        controlled["candidate_label"] != removed_label
    ]
    if len(primary) != 1 or len(original_nonprimary) != 3 or len(controlled_nonprimary) != 3:
        raise RuntimeError("Invalid held-out red/blue/green CE rows.")

    primary_row = primary.iloc[0]
    original_best = original_nonprimary.loc[
        original_nonprimary["mean_causal_effect"].idxmax()
    ]
    controlled_best = controlled_nonprimary.loc[
        controlled_nonprimary["mean_causal_effect"].idxmax()
    ]

    red = float(original_best["mean_causal_effect"])
    blue = float(controlled_best["mean_causal_effect"])
    green = float(primary_row["mean_causal_effect"])

    return {
        "removed_label": removed_label,
        "removed_label_name": LABEL_TO_WORD[removed_label],
        "feature_rank": int(selected.feature_rank),
        "feature_idx": feature_idx,
        "steering_alpha": float(CAUSAL_ALPHA),
        "activation_afr_rank_biserial": float(
            selected.activation_afr_rank_biserial
        ),
        "screen_original_primary_effect": float(
            selected.screen_original_primary_effect
        ),
        "screen_original_strongest_nonprimary_effect": float(
            selected.screen_original_strongest_nonprimary_effect
        ),
        "screen_causal_selectivity_margin": float(
            selected.screen_causal_selectivity_margin
        ),
        "original_strongest_nonprimary_label": int(
            original_best["candidate_label"]
        ),
        "original_strongest_nonprimary_label_name": str(
            original_best["candidate_label_name"]
        ),
        "original_strongest_nonprimary_effect": red,
        "controlled_strongest_nonprimary_label": int(
            controlled_best["candidate_label"]
        ),
        "controlled_strongest_nonprimary_label_name": str(
            controlled_best["candidate_label_name"]
        ),
        "controlled_strongest_nonprimary_effect": blue,
        "original_primary_effect": green,
        "original_primary_positive": bool(green > 0),
        "controlled_nonprimary_positive": bool(blue > 0),
        "nonprimary_transfer_gain": blue - red,
        "nonprimary_increased": bool(blue > red),
        "partial_transfer_ordering": bool(red < blue < green),
        "n_original_prompts": expected_original,
        "n_controlled_prompts": expected_controlled,
    }


def _heldout_dashboard_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    out = summary_df.copy().sort_values(["removed_label", "feature_rank"])
    for target, source in {
        "red_holdout_CE_pp": "original_strongest_nonprimary_effect",
        "blue_holdout_CE_pp": "controlled_strongest_nonprimary_effect",
        "green_holdout_CE_pp": "original_primary_effect",
        "blue_minus_red_holdout_pp": "nonprimary_transfer_gain",
    }.items():
        out[target] = 100.0 * out[source]
    return out[[
        "removed_label_name",
        "feature_rank",
        "feature_idx",
        "screen_causal_selectivity_margin",
        "original_strongest_nonprimary_label_name",
        "red_holdout_CE_pp",
        "controlled_strongest_nonprimary_label_name",
        "blue_holdout_CE_pp",
        "green_holdout_CE_pp",
        "blue_minus_red_holdout_pp",
        "controlled_nonprimary_positive",
        "nonprimary_increased",
        "partial_transfer_ordering",
    ]].rename(columns={
        "removed_label_name": "primary/removed label",
        "feature_rank": "AFR rank",
        "feature_idx": "feature",
        "screen_causal_selectivity_margin": "screen green-red margin",
        "original_strongest_nonprimary_label_name": "red label",
        "controlled_strongest_nonprimary_label_name": "blue label",
        "controlled_nonprimary_positive": "blue > 0",
        "nonprimary_increased": "blue > red",
        "partial_transfer_ordering": "red < blue < green",
    }).round(4)


def _heldout_overall_so_far(
    feature_summary_df: pd.DataFrame,
    final_feature_table: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for label in CAUSAL_REMOVED_LABELS:
        label = int(label)
        sub = feature_summary_df[feature_summary_df["removed_label"] == label]
        total = int((final_feature_table["primary_label"] == label).sum())
        rows.append({
            "primary_label": LABEL_TO_WORD[label],
            "completed": int(len(sub)),
            "total_screened_in": total,
            "progress": f"{len(sub)}/{total}",
            "red_mean_pp": (
                100.0 * float(sub["original_strongest_nonprimary_effect"].mean())
                if len(sub) else np.nan
            ),
            "blue_mean_pp": (
                100.0 * float(sub["controlled_strongest_nonprimary_effect"].mean())
                if len(sub) else np.nan
            ),
            "green_mean_pp": (
                100.0 * float(sub["original_primary_effect"].mean())
                if len(sub) else np.nan
            ),
            "fraction_blue_gt_red": (
                float(sub["nonprimary_increased"].mean()) if len(sub) else np.nan
            ),
        })
    return pd.DataFrame(rows).round(4)


def run_heldout_causal_evaluation(
    prompt_table: pd.DataFrame,
    final_feature_table: pd.DataFrame,
):
    """Evaluate only causally screened-in features on query-disjoint holdout prompts."""
    if set(prompt_table["causal_split"]) != {"holdout"}:
        raise ValueError("Holdout prompt table has the wrong causal_split.")
    if len(final_feature_table) == 0:
        print("No feature passed causal screening; held-out evaluation is empty.")
        return prompt_table.copy(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    scored = _merge_causal_baselines(prompt_table, "held-out")
    long_parts = []
    summaries = []
    total_features = int(len(final_feature_table))

    for feature_number, selected in enumerate(
        final_feature_table.itertuples(index=False),
        start=1,
    ):
        feature_long = _score_one_feature_on_scored_prompts(
            scored,
            selected,
            conditions=["original", "controlled"],
        )
        long_parts.append(feature_long)
        summaries.append(_heldout_feature_summary(feature_long, selected))

        summary_so_far = pd.DataFrame(summaries).sort_values(
            ["removed_label", "feature_rank"]
        ).reset_index(drop=True)
        overall_so_far = _heldout_overall_so_far(
            summary_so_far,
            final_feature_table,
        )
        globals()["causal_heldout_feature_summary_so_far_df"] = summary_so_far.copy()
        globals()["causal_heldout_overall_summary_so_far_df"] = overall_so_far.copy()

        if CAUSAL_SHOW_HOLDOUT_RESULTS:
            if CAUSAL_CLEAR_OUTPUT_BETWEEN_FEATURES:
                clear_output(wait=True)
            latest = summary_so_far.iloc[-1]
            print("=" * 112)
            print(
                f"Held-out causal AFR: {feature_number}/{total_features} | "
                f"label={latest['removed_label_name']} | "
                f"feature={int(latest['feature_idx'])} | "
                f"blue-red={100.0 * float(latest['nonprimary_transfer_gain']):+.3f} pp"
            )
            print("=" * 112)
            display(_heldout_dashboard_table(summary_so_far))
            print("Held-out overall so far")
            display(overall_so_far)

    heldout_long_df = pd.concat(long_parts, ignore_index=True)
    heldout_summary_df = pd.DataFrame(summaries).sort_values(
        ["removed_label", "feature_rank"]
    ).reset_index(drop=True)
    heldout_overall_df = _heldout_overall_so_far(
        heldout_summary_df,
        final_feature_table,
    )
    return scored, heldout_long_df, heldout_summary_df, heldout_overall_df


if RUN_AFR_CAUSAL_EXPERIMENT:
    (
        causal_screening_prompt_scored_df,
        causal_screening_effect_long_df,
        causal_screening_feature_summary_df,
        causal_final_feature_df,
    ) = run_causal_specificity_screen(
        causal_screening_prompt_df,
        causal_selected_feature_df,
    )

    print("Causal screening complete.")
    display(_screening_count_table(causal_screening_feature_summary_df))
    print("Features admitted to held-out evaluation")
    display(causal_final_feature_df)

    (
        causal_heldout_prompt_scored_df,
        causal_heldout_effect_long_df,
        causal_heldout_feature_summary_so_far_df,
        causal_heldout_overall_summary_so_far_df,
    ) = run_heldout_causal_evaluation(
        causal_holdout_prompt_df,
        causal_final_feature_df,
    )

    # Backward-compatible aliases: all final figures/tables use held-out results.
    causal_prompt_scored_df = causal_heldout_prompt_scored_df
    causal_effect_long_df = causal_heldout_effect_long_df
    causal_feature_summary_so_far_df = causal_heldout_feature_summary_so_far_df
    causal_overall_summary_so_far_df = causal_heldout_overall_summary_so_far_df
else:
    causal_screening_prompt_scored_df = causal_screening_prompt_df.copy()
    causal_screening_effect_long_df = pd.DataFrame()
    causal_screening_feature_summary_df = pd.DataFrame()
    causal_final_feature_df = pd.DataFrame()
    causal_heldout_prompt_scored_df = causal_holdout_prompt_df.copy()
    causal_heldout_effect_long_df = pd.DataFrame()
    causal_heldout_feature_summary_so_far_df = pd.DataFrame()
    causal_heldout_overall_summary_so_far_df = pd.DataFrame()
    causal_prompt_scored_df = causal_heldout_prompt_scored_df
    causal_effect_long_df = causal_heldout_effect_long_df
    causal_feature_summary_so_far_df = causal_heldout_feature_summary_so_far_df
    causal_overall_summary_so_far_df = causal_heldout_overall_summary_so_far_df
    print("RUN_AFR_CAUSAL_EXPERIMENT=False; causal forward passes were skipped.")

In [ ]:
# ============================================================
# 21. Aggregate held-out effects and draw the screened-feature bars
# ============================================================


def sem(values: Sequence[float]) -> float:
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values) <= 1:
        return 0.0 if len(values) == 1 else np.nan
    return float(values.std(ddof=1) / math.sqrt(len(values)))


def aggregate_causal_effect_by_feature_and_label(
    long_df: pd.DataFrame,
    *,
    gold_conditioned: bool,
) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame()

    source = long_df.copy()
    if gold_conditioned:
        source = source[source["gold_label"] == source["candidate_label"]].copy()
        aggregation_scope = "gold_label_conditioned"
    else:
        aggregation_scope = "all_prompts_in_condition"

    result = (
        source
        .groupby([
            "causal_split",
            "removed_label",
            "removed_label_name",
            "feature_rank",
            "feature_idx",
            "steering_alpha",
            "condition",
            "candidate_label",
            "candidate_label_name",
        ], as_index=False)
        .agg(
            mean_causal_effect=("delta_probability", "mean"),
            sem_causal_effect=("delta_probability", sem),
            median_causal_effect=("delta_probability", "median"),
            fraction_effect_positive=(
                "delta_probability",
                lambda values: float(np.mean(np.asarray(values) > 0)),
            ),
            n_prompts=("delta_probability", "count"),
        )
    )
    result["aggregation_scope"] = aggregation_scope
    return result


def aggregate_heldout_causal_afr_results(
    long_df: pd.DataFrame,
    final_feature_table: pd.DataFrame,
):
    all_prompt_ce = aggregate_causal_effect_by_feature_and_label(
        long_df,
        gold_conditioned=False,
    )
    gold_conditioned_ce = aggregate_causal_effect_by_feature_and_label(
        long_df,
        gold_conditioned=True,
    )

    if len(final_feature_table) == 0:
        return all_prompt_ce, gold_conditioned_ce, pd.DataFrame(), pd.DataFrame()

    expected_original = 4 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
    expected_controlled = 3 * int(CAUSAL_HOLDOUT_EXAMPLES_PER_LABEL)
    original_counts = set(
        all_prompt_ce.loc[
            all_prompt_ce["condition"] == "original", "n_prompts"
        ].astype(int)
    )
    controlled_counts = set(
        all_prompt_ce.loc[
            all_prompt_ce["condition"] == "controlled", "n_prompts"
        ].astype(int)
    )
    if original_counts != {expected_original}:
        raise RuntimeError(
            f"Every held-out original CE row must use {expected_original} prompts; "
            f"got {sorted(original_counts)}."
        )
    if controlled_counts != {expected_controlled}:
        raise RuntimeError(
            f"Every held-out controlled CE row must use {expected_controlled} prompts; "
            f"got {sorted(controlled_counts)}."
        )

    feature_rows = []
    for selected in final_feature_table.itertuples(index=False):
        removed_label = int(selected.primary_label)
        feature_idx = int(selected.feature_idx)
        feature_ce = all_prompt_ce[
            (all_prompt_ce["removed_label"] == removed_label)
            & (all_prompt_ce["feature_idx"] == feature_idx)
        ]
        original = feature_ce[feature_ce["condition"] == "original"]
        controlled = feature_ce[feature_ce["condition"] == "controlled"]

        primary = original[original["candidate_label"] == removed_label]
        original_nonprimary = original[
            original["candidate_label"] != removed_label
        ]
        controlled_nonprimary = controlled[
            controlled["candidate_label"] != removed_label
        ]
        if len(primary) != 1 or len(original_nonprimary) != 3 or len(controlled_nonprimary) != 3:
            raise RuntimeError("Invalid held-out feature CE structure.")

        primary_row = primary.iloc[0]
        original_best = original_nonprimary.loc[
            original_nonprimary["mean_causal_effect"].idxmax()
        ]
        controlled_best = controlled_nonprimary.loc[
            controlled_nonprimary["mean_causal_effect"].idxmax()
        ]
        red = float(original_best["mean_causal_effect"])
        blue = float(controlled_best["mean_causal_effect"])
        green = float(primary_row["mean_causal_effect"])

        feature_rows.append({
            "removed_label": removed_label,
            "removed_label_name": LABEL_TO_WORD[removed_label],
            "feature_rank": int(selected.feature_rank),
            "feature_idx": feature_idx,
            "selection_rule": str(selected.selection_rule),
            "final_selection_rule": str(selected.final_selection_rule),
            "preactivation_selectivity_margin": float(
                selected.preactivation_selectivity_margin
            ),
            "primary_firing_frequency": float(selected.primary_firing_frequency),
            "activation_afr_type": str(selected.activation_afr_type),
            "activation_afr_rank_biserial": float(
                selected.activation_afr_rank_biserial
            ),
            "activation_afr_mean_shift": float(selected.activation_afr_mean_shift),
            "screen_original_primary_effect": float(
                selected.screen_original_primary_effect
            ),
            "screen_original_strongest_nonprimary_label": int(
                selected.screen_original_strongest_nonprimary_label
            ),
            "screen_original_strongest_nonprimary_label_name": str(
                selected.screen_original_strongest_nonprimary_label_name
            ),
            "screen_original_strongest_nonprimary_effect": float(
                selected.screen_original_strongest_nonprimary_effect
            ),
            "screen_causal_selectivity_margin": float(
                selected.screen_causal_selectivity_margin
            ),
            "causal_screen_pass": bool(selected.causal_screen_pass),
            "steering_alpha": float(selected.steering_alpha),
            "original_strongest_nonprimary_label": int(
                original_best["candidate_label"]
            ),
            "original_strongest_nonprimary_label_name": str(
                original_best["candidate_label_name"]
            ),
            "original_strongest_nonprimary_effect": red,
            "controlled_strongest_nonprimary_label": int(
                controlled_best["candidate_label"]
            ),
            "controlled_strongest_nonprimary_label_name": str(
                controlled_best["candidate_label_name"]
            ),
            "controlled_strongest_nonprimary_effect": blue,
            "original_primary_effect": green,
            "original_primary_positive": bool(green > 0),
            "controlled_nonprimary_positive": bool(blue > 0),
            "nonprimary_transfer_gain": blue - red,
            "nonprimary_increased": bool(blue > red),
            "partial_transfer_ordering": bool(red < blue < green),
            "n_original_prompts": expected_original,
            "n_controlled_prompts": expected_controlled,
        })

    feature_summary = pd.DataFrame(feature_rows).sort_values(
        ["removed_label", "feature_rank"]
    ).reset_index(drop=True)

    overall_rows = []
    for removed_label in CAUSAL_REMOVED_LABELS:
        removed_label = int(removed_label)
        sub = feature_summary[feature_summary["removed_label"] == removed_label]
        n_features = int(len(sub))
        overall_rows.append({
            "removed_label": removed_label,
            "removed_label_name": LABEL_TO_WORD[removed_label],
            "n_features": n_features,
            "mean_activation_afr_rank_biserial": (
                float(sub["activation_afr_rank_biserial"].mean())
                if n_features else np.nan
            ),
            "original_nonprimary_mean": (
                float(sub["original_strongest_nonprimary_effect"].mean())
                if n_features else np.nan
            ),
            "original_nonprimary_sem": (
                sem(sub["original_strongest_nonprimary_effect"])
                if n_features else np.nan
            ),
            "controlled_nonprimary_mean": (
                float(sub["controlled_strongest_nonprimary_effect"].mean())
                if n_features else np.nan
            ),
            "controlled_nonprimary_sem": (
                sem(sub["controlled_strongest_nonprimary_effect"])
                if n_features else np.nan
            ),
            "original_primary_mean": (
                float(sub["original_primary_effect"].mean())
                if n_features else np.nan
            ),
            "original_primary_sem": (
                sem(sub["original_primary_effect"])
                if n_features else np.nan
            ),
            "mean_nonprimary_transfer_gain": (
                float(sub["nonprimary_transfer_gain"].mean())
                if n_features else np.nan
            ),
            "fraction_features_primary_positive": (
                float(sub["original_primary_positive"].mean())
                if n_features else np.nan
            ),
            "fraction_features_controlled_nonprimary_positive": (
                float(sub["controlled_nonprimary_positive"].mean())
                if n_features else np.nan
            ),
            "fraction_features_nonprimary_increases": (
                float(sub["nonprimary_increased"].mean())
                if n_features else np.nan
            ),
            "fraction_features_partial_transfer_ordering": (
                float(sub["partial_transfer_ordering"].mean())
                if n_features else np.nan
            ),
            "n_original_prompts_per_feature": expected_original,
            "n_controlled_prompts_per_feature": expected_controlled,
        })

    overall_summary = pd.DataFrame(overall_rows).sort_values(
        "removed_label"
    ).reset_index(drop=True)
    return all_prompt_ce, gold_conditioned_ce, feature_summary, overall_summary


if RUN_AFR_CAUSAL_EXPERIMENT:
    (
        causal_effect_by_feature_and_label_df,
        causal_gold_conditioned_effect_by_feature_and_label_df,
        causal_feature_summary_df,
        causal_overall_summary_df,
    ) = aggregate_heldout_causal_afr_results(
        causal_heldout_effect_long_df,
        causal_final_feature_df,
    )

    print("Causal-screening results for all 40 activation-AFR features")
    display(_screening_dashboard_table(causal_screening_feature_summary_df))
    print("Features retained for the held-out figure")
    display(causal_final_feature_df)
    print("Held-out per-feature causal role-reorganization summary")
    display(causal_feature_summary_df)
    print("Held-out main-bar summary")
    display(causal_overall_summary_df)

    # Held-out baseline accuracy diagnostic.
    baseline_probability_columns = [
        f"baseline_prob_{LABEL_TO_WORD[label]}" for label in CAUSAL_LABEL_ID_ORDER
    ]
    causal_heldout_prompt_scored_df = causal_heldout_prompt_scored_df.copy()
    causal_heldout_prompt_scored_df["baseline_pred_label"] = np.argmax(
        causal_heldout_prompt_scored_df[baseline_probability_columns].to_numpy(
            dtype=float
        ),
        axis=1,
    )
    causal_heldout_prompt_scored_df["baseline_correct"] = (
        causal_heldout_prompt_scored_df["baseline_pred_label"]
        == causal_heldout_prompt_scored_df["gold_label"]
    )
    causal_baseline_accuracy_df = (
        causal_heldout_prompt_scored_df
        .groupby(["removed_label_name", "condition"])
        .agg(
            accuracy=("baseline_correct", "mean"),
            n=("baseline_correct", "size"),
        )
        .reset_index()
    )
    print("Held-out baseline label-normalized accuracy diagnostic")
    display(causal_baseline_accuracy_df)

    # Final grouped bar plot: only screened-in features, evaluated on holdout.
    x_labels = [
        f"{row.removed_label_name}"
        for row in causal_overall_summary_df.itertuples(index=False)
    ]
    red_pp = 100.0 * causal_overall_summary_df[
        "original_nonprimary_mean"
    ].to_numpy(dtype=float)
    blue_pp = 100.0 * causal_overall_summary_df[
        "controlled_nonprimary_mean"
    ].to_numpy(dtype=float)
    green_pp = 100.0 * causal_overall_summary_df[
        "original_primary_mean"
    ].to_numpy(dtype=float)

    def _bar_text(values):
        return ["" if not np.isfinite(value) else f"{value:.2f}%" for value in values]

    causal_overall_fig = go.Figure()
    causal_overall_fig.add_trace(go.Bar(
        x=x_labels,
        y=red_pp,
        name="Non-Primary Labels (Baseline)",
        marker=dict(color="#cc7474", line=dict(color="#9b2424", width=2)),
        text=_bar_text(red_pp),
        textposition="outside",
        hovertemplate=(
            "Primary/removed label=%{x}<br>"
            "Held-out mean strongest non-primary CE=%{y:.2f} pp"
            "<extra></extra>"
        ),
    ))
    causal_overall_fig.add_trace(go.Bar(
        x=x_labels,
        y=blue_pp,
        name="Non-Primary Labels (Reorganized)",
        marker=dict(color="#5b79db", line=dict(color="#263ba5", width=2)),
        text=_bar_text(blue_pp),
        textposition="outside",
        hovertemplate=(
            "Primary/removed label=%{x}<br>"
            "Held-out mean strongest retained-label CE=%{y:.2f} pp"
            "<extra></extra>"
        ),
    ))
    causal_overall_fig.add_trace(go.Bar(
        x=x_labels,
        y=green_pp,
        name="Primary Label (Baseline)",
        marker=dict(color="#59b884", line=dict(color="#3c9367", width=2)),
        text=_bar_text(green_pp),
        textposition="outside",
        hovertemplate=(
            "Primary/removed label=%{x}<br>"
            "Held-out mean primary-label CE=%{y:.2f} pp"
            "<extra></extra>"
        ),
    ))

    causal_overall_fig.add_hline(y=0.0, line_width=3, line_color="black")
    causal_overall_fig.update_layout(
        barmode="group",
        title=dict(
            text=(
                "Adaptive Feature Reorganization (Causal) in LLMs"
                f"<br><sup>{MODEL_SHORT_NAME} | alpha={float(CAUSAL_ALPHA):g}</sup>"
            ),
            x=0.5,
            font=dict(family="Times New Roman", size=34),
        ),
        xaxis=dict(
            title="Causal L-Specific, Strong-AFR Features",
            title_font=dict(family="Times New Roman", size=27),
            tickfont=dict(family="Times New Roman", size=23),
        ),
        yaxis=dict(
            title="Average Change in P(Label)",
            title_font=dict(family="Times New Roman", size=27),
            tickfont=dict(family="Times New Roman", size=20),
            zeroline=False,
        ),
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=1.03,
            yanchor="bottom",
            font=dict(family="Times New Roman", size=19),
            bgcolor="rgba(255,255,255,0.90)",
            bordercolor="#294b4b",
            borderwidth=1,
        ),
        width=1500,
        height=800,
        margin=dict(l=130, r=55, t=195, b=120),
        font=dict(family="Times New Roman", color="#243b63"),
#         template="plotly_white",
        uniformtext_minsize=14,
        uniformtext_mode="show",
    )
    causal_overall_fig.show()


def show_causal_feature_transfer(
    removed_label: int,
    feature_idx: int,
    *,
    show_screening: bool = True,
    show_gold_conditioned: bool = True,
) -> pd.DataFrame:
    """Inspect screening and held-out CE for one screened-in feature."""
    removed_label = int(removed_label)
    feature_idx = int(feature_idx)

    if show_screening:
        screen = causal_screening_feature_summary_df[
            (causal_screening_feature_summary_df["removed_label"] == removed_label)
            & (causal_screening_feature_summary_df["feature_idx"] == feature_idx)
        ].copy()
        print("Independent causal-screening result")
        display(screen)

    subset = causal_effect_by_feature_and_label_df[
        (causal_effect_by_feature_and_label_df["removed_label"] == removed_label)
        & (causal_effect_by_feature_and_label_df["feature_idx"] == feature_idx)
    ].copy()
    if subset.empty:
        raise ValueError(
            f"No held-out result for removed_label={removed_label}, feature={feature_idx}. "
            "The feature may not have passed causal screening."
        )
    subset["effect_percentage_points"] = 100.0 * subset["mean_causal_effect"]
    subset = subset.sort_values(["condition", "candidate_label"]).reset_index(drop=True)
    print("Held-out all-prompt CE (main definition)")
    display(subset)

    if show_gold_conditioned:
        diagnostic = causal_gold_conditioned_effect_by_feature_and_label_df[
            (causal_gold_conditioned_effect_by_feature_and_label_df["removed_label"] == removed_label)
            & (causal_gold_conditioned_effect_by_feature_and_label_df["feature_idx"] == feature_idx)
        ].copy()
        diagnostic["effect_percentage_points"] = (
            100.0 * diagnostic["mean_causal_effect"]
        )
        print("Held-out gold-label-conditioned CE (secondary diagnostic)")
        display(diagnostic.sort_values(["condition", "candidate_label"]))

    fig = go.Figure()
    for condition in ["original", "controlled"]:
        condition_subset = subset[subset["condition"] == condition]
        fig.add_trace(go.Bar(
            x=condition_subset["candidate_label_name"],
            y=condition_subset["effect_percentage_points"],
            name=condition,
            text=[
                f"{value:+.2f}"
                for value in condition_subset["effect_percentage_points"]
            ],
            textposition="auto",
        ))
    fig.add_hline(y=0.0, line_dash="dash")
    fig.update_layout(
        barmode="group",
        title=(
            f"Feature {feature_idx}: held-out causal effect after removing "
            f"{LABEL_TO_WORD[removed_label]}"
        ),
        xaxis_title="Candidate label c",
        yaxis_title="Mean change in P(c) (percentage points)",
        width=950,
        height=600,
        font=dict(family="Times New Roman", size=16),
    )
    fig.show()
    return subset